# Análisis de backbones — Queratocono (Orbscan) · Sano (0) vs Queratocono (1)

**Proyecto Final Integrador** · Etapa 02, posterior al EDA (`01_EDA_queratocono.ipynb`).

**Objetivo de decisión.** Clasificar cada **ojo** como *sano* (0) o *queratocono* (1) a partir de los
4 mapas Orbscan (Anterior, Axial, Pachymetry, Posterior), con una prioridad explícita: **no dejar
pasar un queratocono** (falso negativo). Un QC no detectado puede terminar en cirugía refractiva
y ectasia, mientras que un falso positivo cuesta un estudio de más.

> ⚠️ **Por defecto se usa solo el 20 % de los pacientes** (`FRACCION_DATOS = 0.2`, en la celda de
> configuración de la Fase 0; se aplica en la Fase 1.1). De ese 20 %, el **80 % es train**
> (validación cruzada con los 5 folds oficiales) y el **20 % es test**. Para la corrida final: `FRACCION_DATOS = 1.0`.

## Qué se prueba
| Fase | Contenido |
|---|---|
| 1 | Manifest por **ojo** (1,407), corrección `3E+132`→`3E132`, asserts anti-fuga, **subset del 20 %**, **qué son los folds** y **evaluación balanceada** |
| 2 | Copia del dataset **sin las letras T/N** (`ORBSCAN_Dataset_modificado`) e inspección OD/OS |
| 3 | Infraestructura: métricas, umbral `tau`, matrices de confusión con conteos y resaltado del elegido |
| 4 | **Baseline tabular**: regresión logística con las variables clínicas (vara de comparación, sin imágenes) |
| 5 | **Análisis de backbones**: transfer learning (solo la última capa) de **MobileNetV2, ResNet50 y EfficientNet-B7**, por mapa + fusión; validación y test |
| 6 | **Fine-tuning** del **15 % final de las capas** del mejor backbone, **con aumento de datos** |
| 7 | **Embeddings + ML clásico** (LogReg, SVM, Random Forest, XGBoost, MLP) |
| 8 | Tabla final, **modelo elegido** y curvas ROC |
| 9 | **Grad-CAM** (qué mira el modelo), con la explicación de los colores |
| 10 | Resumen, exportación y **tabla de tiempos** por etapa |

## Métricas
* **Selección** (en todas las tablas el elegido va **en verde** y la columna del criterio tiene el encabezado verde): **AUC-ROC out-of-fold (OOF)** redondeado a 3 decimales; si empatan, **especificidad OOF @tau**. El test **nunca** participa de la elección.
* **Punto de operación**: `tau` = mayor umbral con **sensibilidad OOF ≥ 0.95**. Se predice QC si `P(QC) ≥ tau`.
* **Reporte en test**: sensibilidad @tau (IC Wilson 95 %) + especificidad @tau, y AUC con IC bootstrap agrupado por paciente. Secundarias: AP, balanced accuracy, F2 y Brier.
* **Evaluación siempre balanceada**: se entrena con todos los ojos, pero validación y test se evalúan sobre un subconjunto con **igual cantidad de sanos y de QC** (semilla fija). Las matrices de confusión muestran **conteos `aciertos/total de la clase real`** y un **pie** con cuántos ojos (e imágenes) se evaluaron por clase.

## Ajustes respecto del pipeline de Mandioca
* **Splits oficiales** del dataset (`split_80_20` + `fold` 1-5, agrupados por paciente): CV 5-fold sobre train, test intacto.
* **Sin letras T/N**: se usa una copia de las imágenes con las esquinas inferiores tapadas (Fase 2).
* **Aumento de datos solo en el fine-tuning** (Fase 6): `RandomResizedCrop(0.85-1)` + rotación ±5° + traslación 3 %. **Sin** volteo vertical (la asimetría inferior/superior es un signo de QC), **sin** rotación grande y **sin** cambios de color (el color codifica el valor del mapa).
* **Ojos OS espejados** horizontalmente (`ESPEJAR_OS`): así todos los ojos quedan con la misma orientación anatómica.
* Problema binario: un logit + `BCEWithLogits(pos_weight = n_sanos / n_QC del fold)`.

## Cómo ejecutar
* **Colab (corrida completa)**: *Entorno de ejecución → GPU* y *Ejecutar todo*. Los datos se descargan solos de Zenodo (CornOrb, registro abierto 20542091, ~680 MB, con verificación md5) a `/content` y se descomprimen allí. Las salidas quedan en `MyDrive/PFI_queratocono/salidas/02_backbone_v2/` y se reutilizan como caché si se corta la sesión (`USAR_DRIVE = False` para no montar Drive).
* **Local**: si no encuentra los datos en `DATA_DIR`, `Datos/zenodo` o `../../..`, los descarga a `Datos/zenodo` (ignorada por git).
* **Fracción de datos**: `FRACCION_DATOS` (celda de configuración) define qué parte de los pacientes se usa. Por defecto 0.2 para pruebas rápidas, 1.0 para la corrida final. Cada fracción guarda su caché aparte.
* **Smoke test** (corrida mínima para validar que nada falle): variable de entorno `SMOKE=1`, o `SMOKE = True` en la celda siguiente. Usa 2 folds, 6 ojos por clase, pocas épocas y los 3 backbones a 128 px. Los artefactos llevan el sufijo `_smoke`.
* **Tiempo orientativo** (Colab T4, 20 %): Fases 1-5 ≈ 10-15 min; el fine-tuning depende del backbone ganador (MobileNetV2 ≈ 5 min, ResNet50 ≈ 10 min, B7 a 600 px ≈ 40 min; se recorta con `MAPAS_FT`); ML clásico ≈ 5-15 min. La Fase 10 muestra los tiempos reales.

## Fase 0 — Entorno, semillas y configuración

In [ ]:
import time
T_INICIO = time.time()                         # para la tabla de tiempos (Fase 10)

import os, sys, json, math, random, zipfile, subprocess, warnings
from importlib.util import find_spec
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    faltan_pkgs = [p for p in ("timm", "xgboost") if find_spec(p) is None]
    if faltan_pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan_pkgs], check=True)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms as T
from PIL import Image, ImageDraw
from scipy import ndimage

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from matplotlib.ticker import MaxNLocator
try:
    from xgboost import XGBClassifier
except ImportError:                              # fallback: HistGradientBoosting de sklearn
    XGBClassifier = None

warnings.filterwarnings("ignore", category=UserWarning, module="timm")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SMOKE = os.environ.get("SMOKE", "0") == "1"    # para forzar en Colab: SMOKE = True
SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 2 if EN_COLAB else 0
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("entorno:", "Colab" if EN_COLAB else "local", "| torch", torch.__version__, "| timm", timm.__version__,
      "| xgboost", "sí" if XGBClassifier is not None else "no (se usa HistGradientBoosting)")
print("device:", DEVICE, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("modo SMOKE:", SMOKE)

In [ ]:
OBJ_SENS = 0.95                      # sensibilidad exigida (out-of-fold) al calibrar tau
MAPAS = ["Anterior", "Axial", "Pachymetry", "Posterior"]
BACKBONES = {                        # nombre timm -> tamaño de entrada nativo
    "mobilenetv2_100.ra_in1k": 224,      # 3.5 M parámetros
    "resnet50.a1_in1k": 224,             # 25.6 M
    "tf_efficientnet_b7.ra_in1k": 600,   # 66 M (la más pesada)
}
CORTO = {"mobilenetv2_100.ra_in1k": "mnv2", "resnet50.a1_in1k": "rn50", "tf_efficientnet_b7.ra_in1k": "effb7"}
FOLDS = [1, 2, 3, 4, 5]
ESPEJAR_OS = True                    # OS (N-T) -> misma orientación que OD (T-N)
LIMPIAR_MARCAS = True                # True: se usa ORBSCAN_Dataset_modificado (sin letras T/N)
HEAD_CFG = dict(epochs=60, patience=10, batch=32, lr=1e-3, weight_decay=1e-4)   # Fase 5: última capa
FT_CFG = dict(epochs=15, patience=5, batch=16, aug=True, pct_capas=0.15,        # Fase 6: fine-tuning
              lr_backbone=1e-4, lr_head=3e-4, weight_decay=1e-2)
FT_BATCH = {"effb7": 8}              # B7 a 600 px: batch más chico para la memoria de una T4
MAPAS_FT = list(MAPAS)               # mapas a los que se aplica fine-tuning (recortar si falta tiempo de GPU)
CRITERIO, DESEMPATE = "AUC_oof", "Esp_oof"     # regla única de elección en todo el notebook
C_LR_TAB = 1.0                       # regularización L2 del baseline tabular (fija)
N_BOOT = 1000                        # bootstrap agrupado por paciente
GRADCAM_POR_CLASE = 2
N_SMOKE = 6                          # ojos por clase y por fold en modo SMOKE
ML_CFG = dict(rf_arboles=300, xgb_arboles=300, pca_comp=128, mlp_early_stopping=True)

# ======================================================================================
# ⚠️  FRACCIÓN DEL DATASET QUE SE USA  ⚠️
# Fracción de PACIENTES que se usa (1.0 = dataset completo). Se muestrea dentro de cada
# fold y del test, por clase, así se mantiene la división oficial 80/20 y no hay fugas.
# Con 0.2: se usa ~20 % de los pacientes; de ese 20 %, el 80 % es train (5 folds) y el 20 % test.
# ======================================================================================
FRACCION_DATOS = 0.2

if SMOKE:
    BACKBONES = {k: 128 for k in BACKBONES}
    FOLDS = [1, 2]
    HEAD_CFG.update(epochs=3, patience=2, batch=8)
    FT_CFG.update(epochs=1, patience=1, batch=4)
    FT_BATCH = {}
    N_BOOT, GRADCAM_POR_CLASE = 50, 1
    ML_CFG.update(rf_arboles=50, xgb_arboles=30, mlp_early_stopping=False)

TAG = "_smoke" if SMOKE else (f"_frac{round(FRACCION_DATOS * 100)}" if FRACCION_DATOS < 1 else "")
if not LIMPIAR_MARCAS:
    TAG += "_conTN"
# el TAG separa la caché (res/, emb/, ckpt/) de cada fracción para no mezclar resultados
print(json.dumps(dict(BACKBONES=BACKBONES, FOLDS=FOLDS, HEAD_CFG=HEAD_CFG, FT_CFG=FT_CFG,
                      OBJ_SENS=OBJ_SENS, N_BOOT=N_BOOT, FRACCION_DATOS=FRACCION_DATOS,
                      LIMPIAR_MARCAS=LIMPIAR_MARCAS, TAG=TAG), indent=1))
if not SMOKE and FRACCION_DATOS < 1:
    print(f"\n⚠️  SE USA SOLO EL {FRACCION_DATOS:.0%} DE LOS PACIENTES (FRACCION_DATOS = {FRACCION_DATOS})")

In [ ]:
import hashlib
import urllib.request

# CornOrb en Zenodo. La versión 17127265 es de acceso restringido; esta (20542091) es abierta,
# CC BY 4.0, con los mismos archivos. Los md5 son los que publica Zenodo.
ZENODO_RECORD = "20542091"
ZENODO_ARCHIVOS = {
    "clinical_data_and_labels.csv": "983fda79c62fbe54d7fa9e83f440cfee",
    "ORBSCAN_Dataset.zip": "4afca93d00abf1c5e1fe25ea681a4e17",   # ~680 MB
}
# En Colab, Drive solo guarda las salidas (caché de resultados y checkpoints) para que
# sobrevivan si se corta la sesión. Con False todo queda en /content y se pierde al cerrar.
USAR_DRIVE = True


def md5(path, bloque=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


def descargar_zenodo(destino):
    """Descarga el CSV y el zip de CornOrb a `destino` y verifica su md5. Si ya están, no hace nada."""
    destino.mkdir(parents=True, exist_ok=True)
    for nombre, esperado in ZENODO_ARCHIVOS.items():
        f = destino / nombre
        if f.exists() and md5(f) == esperado:
            continue
        url = f"https://zenodo.org/api/records/{ZENODO_RECORD}/files/{nombre}/content"
        print(f"descargando {nombre} de Zenodo ...")
        tmp = f.with_suffix(f.suffix + ".part")
        with urllib.request.urlopen(url) as r, open(tmp, "wb") as out:
            total = int(r.headers.get("Content-Length", 0))
            leido, ultimo = 0, -1
            while chunk := r.read(1 << 22):
                out.write(chunk)
                leido += len(chunk)
                pct = int(100 * leido / total) if total else 0
                if pct // 10 != ultimo:
                    ultimo = pct // 10
                    print(f"  {pct:3d} % ({leido / 1e6:.0f} MB)")
        assert md5(tmp) == esperado, f"md5 incorrecto en {nombre}: borrar {tmp} y reintentar"
        tmp.replace(f)
    return destino


def es_dir_datos(p):
    return p is not None and (p / "clinical_data_and_labels.csv").exists() and \
        ((p / "ORBSCAN_Dataset").is_dir() or (p / "ORBSCAN_Dataset.zip").exists())


if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Primero se busca una copia ya descargada; si no hay, se baja de Zenodo
CANDIDATOS = [os.environ.get("DATA_DIR"), "Datos/zenodo", "../Datos/zenodo", "../../..",
              "/content/datos_zenodo", "/content/drive/MyDrive/PFI_queratocono"]
DATA_DIR = next((Path(c).resolve() for c in CANDIDATOS if c and es_dir_datos(Path(c))), None)
if DATA_DIR is None:
    DATA_DIR = descargar_zenodo(Path("/content/datos_zenodo") if EN_COLAB else Path("Datos/zenodo").resolve())
CSV_PATH = DATA_DIR / "clinical_data_and_labels.csv"

IMG_ROOT = DATA_DIR / "ORBSCAN_Dataset"
if EN_COLAB and (DATA_DIR / "ORBSCAN_Dataset.zip").exists():
    IMG_ROOT = Path("/content/ORBSCAN_Dataset")       # disco local de Colab: mucho más rápido que Drive
    if not IMG_ROOT.is_dir():
        with zipfile.ZipFile(DATA_DIR / "ORBSCAN_Dataset.zip") as z:
            z.extractall("/content")
elif not IMG_ROOT.is_dir():
    with zipfile.ZipFile(DATA_DIR / "ORBSCAN_Dataset.zip") as z:
        z.extractall(DATA_DIR)
assert IMG_ROOT.is_dir(), IMG_ROOT
IMG_ROOT_ORIG = IMG_ROOT                            # imágenes originales (con letras T/N)

if os.environ.get("OUT_DIR"):
    OUT = Path(os.environ["OUT_DIR"])
elif EN_COLAB:
    base = Path("/content/drive/MyDrive/PFI_queratocono") if USAR_DRIVE else Path("/content")
    OUT = base / "salidas" / "02_backbone_v2"
else:
    OUT = Path("salidas") / "02_backbone_v2"
for sub in ("res", "ckpt", "emb", "fig"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print("datos  :", DATA_DIR)
print("mapas  :", IMG_ROOT)
print("salidas:", OUT.resolve())

In [ ]:
# ---- cronómetro por etapa (tabla final en la Fase 10) ----
# Cada etapa mide su tiempo real. Lo que se recupera de la caché (embeddings, modelos ya
# entrenados) suma el tiempo de cómputo que se había guardado, así la tabla refleja lo que
# cuesta la etapa completa aunque la sesión se haya cortado y retomado.
TIEMPOS_F = OUT / f"tiempos{TAG}.json"
try:
    TIEMPOS = json.loads(TIEMPOS_F.read_text(encoding="utf-8")) if TIEMPOS_F.exists() else {}
except ValueError:
    TIEMPOS = {}
_ABIERTAS = {}


def fmt_t(s):
    s = float(s)
    if s < 60:
        return f"{s:.1f} s"
    h, resto = divmod(int(round(s)), 3600)
    m, s = divmod(resto, 60)
    return f"{h} h {m:02d} min {s:02d} s" if h else f"{m} min {s:02d} s"


def registrar(nombre, real_s, cache_s=0.0, padre=None):
    TIEMPOS.pop(nombre, None)
    TIEMPOS[nombre] = dict(etapa=nombre, padre=padre, real_s=round(real_s, 1), cache_s=round(cache_s, 1),
                           total_s=round(real_s + cache_s, 1))
    TIEMPOS_F.write_text(json.dumps(TIEMPOS, indent=1, ensure_ascii=False), encoding="utf-8")
    extra = f" (incluye {fmt_t(cache_s)} de cómputo recuperado de caché)" if cache_s > 0.05 else ""
    print(f"⏱ {nombre}: {fmt_t(real_s + cache_s)}{extra}")


def iniciar_etapa(nombre, padre=None):
    _ABIERTAS[nombre] = dict(t0=time.time(), cache_s=0.0, padre=padre)


def cerrar_etapa(nombre):
    d = _ABIERTAS.pop(nombre, None)
    if d is not None:
        registrar(nombre, time.time() - d["t0"], d["cache_s"], d["padre"])


def desde_cache(segundos):
    for d in _ABIERTAS.values():
        d["cache_s"] += float(segundos or 0)


class subetapa:
    """Detalle dentro de una etapa (p. ej. un backbone de la Fase 5)."""
    def __init__(self, nombre, padre):
        self.nombre, self.padre = nombre, padre

    def __enter__(self):
        iniciar_etapa(self.nombre, self.padre)
        return self

    def __exit__(self, tipo, *_):
        if tipo is None:
            cerrar_etapa(self.nombre)
        else:
            _ABIERTAS.pop(self.nombre, None)
        return False


registrar("Fase 0 — Entorno, configuración y datos crudos", time.time() - T_INICIO)

## Fase 1 — Datos: manifest por ojo, subset, folds y evaluación balanceada

* **Unidad de muestra = ojo.** El CSV tiene 1,454 filas, pero solo hay 1,407 carpetas de ojo con 4 PNG. Las filas repetidas son mediciones del mismo ojo que **comparten las mismas imágenes**. Se agrupa por `(patient_code, eye)` y las variables clínicas se promedian (solo se usan en el baseline tabular).
* `3E+132` es la versión de Excel de `3E132` (la carpeta). Se corrige antes del join.
* Variables clínicas: los Kmax < 30 D (4.0 / 4.9 / 5.7, físicamente imposibles) pasan a NaN y se imputan con la mediana del train de cada fold. Los ojos con paquimetría mínima > central se marcan y se conservan. Los ejes (0-180°) se codifican como `sin(2θ)` y `cos(2θ)`.

In [ ]:
iniciar_etapa("Fase 1 — Datos, subset y folds")
df_raw = pd.read_csv(CSV_PATH, dtype={"patient_code": str})
corr = df_raw["patient_code"].str.contains("E+", regex=False)
print("códigos con notación científica corregidos:", sorted(df_raw.loc[corr, "patient_code"].unique()))
df_raw["patient_code"] = df_raw["patient_code"].str.replace("E+", "E", regex=False)

df_raw["kmax_invalido"] = df_raw["kmax_value_D"] < 30
df_raw.loc[df_raw["kmax_invalido"], "kmax_value_D"] = np.nan
df_raw["flag_pachy"] = df_raw["pachy_thinnest_um"] > df_raw["pachy_central_um"]
for col, nom in [("astig_axis_deg", "astig"), ("kmax_axis_deg", "kmax")]:
    rad = np.deg2rad(2 * df_raw[col])
    df_raw[f"{nom}_sin"], df_raw[f"{nom}_cos"] = np.sin(rad), np.cos(rad)
print(f"filas CSV: {len(df_raw)} | Kmax inválidos -> NaN: {int(df_raw.kmax_invalido.sum())} "
      f"| paquimetría mínima > central: {int(df_raw.flag_pachy.sum())}")

VARS_TAB = ["age_years", "astig_value_D", "kmax_value_D", "pachy_central_um", "pachy_thinnest_um",
            "pachy_thinnest_x", "pachy_thinnest_y", "asphericity_anterior", "asphericity_posterior",
            "astig_sin", "astig_cos", "kmax_sin", "kmax_cos"]

g = df_raw.groupby(["patient_code", "eye"], sort=True)
consist = g[["label", "split_80_20", "fold", "gender"]].nunique(dropna=False)
assert (consist <= 1).all().all(), "label/split/fold/gender no son constantes dentro de un ojo"

man = g.agg(**{c: (c, "mean") for c in VARS_TAB},
            label=("label", "first"), gender=("gender", "first"),
            split=("split_80_20", "first"), fold=("fold", "first"),
            n_mediciones=("label", "size"), kmax_invalido=("kmax_invalido", "any"),
            flag_pachy=("flag_pachy", "any")).reset_index()
man["fold"] = man["fold"].fillna(0).astype(int)          # 0 = test
man["label"] = man["label"].astype(int)
man["eye_key"] = man["patient_code"] + "_" + man["eye"]
for m in MAPAS:
    man[f"rel_{m}"] = man.patient_code + "/" + man.eye + "/" + man.patient_code + "_" + man.eye + "_" + m + ".png"

# --- verificaciones ---
carpetas = {(p.parent.name, p.name) for p in IMG_ROOT.glob("*/*") if p.is_dir()}
assert carpetas == set(zip(man.patient_code, man.eye)), "CSV y carpetas no coinciden"
faltan = [r for m in MAPAS for r in man[f"rel_{m}"] if not (IMG_ROOT / r).exists()]
assert not faltan, f"faltan {len(faltan)} PNG, p.ej. {faltan[:3]}"
assert len(man) == 1407, len(man)
pac = man.groupby("patient_code").agg(n_split=("split", "nunique"), n_fold=("fold", "nunique"),
                                      n_lab=("label", "nunique"))
assert (pac.n_split == 1).all(), "paciente en train y test a la vez (fuga)"
assert (pac.n_fold == 1).all(), "paciente en más de un fold (fuga entre folds)"
assert (pac.n_lab == 1).all()
assert set(man.loc[man.split == "train", "fold"]) == {1, 2, 3, 4, 5}
assert (man.loc[man.split == "test", "fold"] == 0).all()

print(f"OK: {len(man)} ojos / {man.patient_code.nunique()} pacientes | "
      f"{int((man.n_mediciones > 1).sum())} ojos con mediciones repetidas (promediadas)")
display(pd.crosstab(man["split"].where(man.split == "test", "fold " + man.fold.astype(str)),
                    man["label"].map({0: "sano", 1: "QC"}), margins=True))
man.drop(columns=[f"rel_{m}" for m in MAPAS]).to_csv(OUT / f"manifest_ojos{TAG}.csv", index=False)

### 1.1 ⚠️ Subset: acá se aplica el 20 % del dataset

La fracción se **declara** en la celda de configuración de la Fase 0 (`FRACCION_DATOS = 0.2`) y se
**aplica** en la celda siguiente:

* se toma el 20 % de los **pacientes** dentro de cada grupo *split oficial × fold × clase*, con semilla 42. Al muestrear **pacientes** (y no ojos) los dos ojos de una persona quedan siempre juntos;
* como se muestrea dentro de cada split oficial, el subset **conserva la proporción 80 % train / 20 % test** y los 5 folds oficiales;
* el 80 % restante de los pacientes **no se usa en ningún paso** (ni para entrenar ni para medir).

En modo SMOKE se reemplaza por 6 ojos por clase y por fold.

In [ ]:
MAN_FULL = man.copy()
if SMOKE:
    partes = [man[(man.split == "train") & (man.fold == k)].groupby("label").sample(n=N_SMOKE, random_state=SEED)
              for k in FOLDS]
    partes.append(man[man.split == "test"].groupby("label").sample(n=N_SMOKE, random_state=SEED))
    man = pd.concat(partes)
else:
    man = man[(man.split == "test") | man.fold.isin(FOLDS)]
    if FRACCION_DATOS < 1:
        pac = man.drop_duplicates("patient_code")[["patient_code", "split", "fold", "label"]]
        elegidos = pac.groupby(["split", "fold", "label"]).sample(frac=FRACCION_DATOS, random_state=SEED)
        man = man[man.patient_code.isin(elegidos.patient_code)]
        print(f"⚠️ FRACCION_DATOS = {FRACCION_DATOS}: {len(elegidos)} de {len(pac)} pacientes")
man = man.sort_values(["patient_code", "eye"]).reset_index(drop=True)

pac_sub = man.groupby("patient_code").agg(n_split=("split", "nunique"), n_fold=("fold", "nunique"))
assert (pac_sub.n_split == 1).all() and (pac_sub.n_fold == 1).all(), "fuga en el subset"

Y = man["label"].values.astype(int)
IDX_POOL = np.where(man.split == "train")[0]          # train (CV) -> predicciones out-of-fold
FOLD_POOL = man["fold"].values[IDX_POOL]
IDX_TEST = np.where(man.split == "test")[0]
Y_POOL, Y_TEST = Y[IDX_POOL], Y[IDX_TEST]
PAC_TEST = man["patient_code"].to_numpy(dtype=str)[IDX_TEST]
print(f"ojos en uso: {len(man)} de {len(MAN_FULL)} ({len(man) / len(MAN_FULL):.0%}) | "
      f"train (CV, {len(FOLDS)} folds): {len(IDX_POOL)} (QC={int(Y_POOL.sum())}) | "
      f"test: {len(IDX_TEST)} (QC={int(Y_TEST.sum())})")

# gráfico: dataset completo vs subset en uso
filas = []
nom_sub = "SMOKE" if SMOKE else f"subset en uso ({min(FRACCION_DATOS, 1):.0%})"
for nom, d in [("dataset completo", MAN_FULL), (nom_sub, man)]:
    for sp in ("train", "test"):
        filas.append(dict(conjunto=f"{nom} · {sp}", Sano=int(((d.split == sp) & (d.label == 0)).sum()),
                          QC=int(((d.split == sp) & (d.label == 1)).sum())))
tab_sub = pd.DataFrame(filas).set_index("conjunto")
ax = tab_sub.iloc[::-1].plot(kind="barh", stacked=True, figsize=(9, 3.4), color=["#4c72b0", "#dd8452"])
for i, tot in enumerate(tab_sub.iloc[::-1].sum(axis=1)):
    ax.text(tot + 5, i, f"{tot} ojos", va="center", fontsize=9)
ax.set_xlabel("ojos")
ax.set_title(f"Qué parte del dataset se usa (train = {len(FOLDS)} folds de CV, test = 20 % oficial)")
plt.tight_layout()
plt.savefig(OUT / "fig" / f"subset{TAG}.png", dpi=100)
plt.show()

### 1.2 ¿Qué son los folds y qué contienen?

Un **fold** es una de las **5 partes** en que los autores del dataset dividieron el **train** (la
columna `fold` del CSV, 1 a 5). Cada fold contiene ojos sanos y con QC en proporciones parecidas,
y **todos los ojos de un mismo paciente caen en el mismo fold**.

La **validación cruzada de 5 folds** entrena 5 veces (5 *rondas*): en la ronda *k* el modelo
aprende con los otros 4 folds y se evalúa con el fold *k*, que **no vio**. Al terminar, **cada
ojo de train tiene una predicción hecha por un modelo que no lo vio**: son las predicciones
**out-of-fold (OOF)**. Con ellas se elige el modelo y se calibra `tau`.

El **test** (`fold = 0` en el manifest) queda aparte: se predice con el **promedio de los 5
modelos** de las rondas y solo se usa para reportar.

**¿Por qué agrupados por paciente?** Los dos ojos de una persona se parecen mucho. Si uno estuviera
en train y el otro en validación, el modelo "reconocería" al paciente y la métrica saldría
optimista (fuga de información).

In [ ]:
filas_f = []
for k in FOLDS + [0]:
    d = man[man.fold == k] if k else man[man.split == "test"]
    filas_f.append({"conjunto": f"fold {k}" if k else "test", "pacientes": d.patient_code.nunique(),
                    "ojos": len(d), "sanos": int((d.label == 0).sum()), "QC": int((d.label == 1).sum()),
                    "% QC": round(100 * d.label.mean(), 1)})
tab_f = pd.DataFrame(filas_f)
tab_f.loc[len(tab_f)] = {"conjunto": "total", "pacientes": man.patient_code.nunique(), "ojos": len(man),
                         "sanos": int((man.label == 0).sum()), "QC": int((man.label == 1).sum()),
                         "% QC": round(100 * man.label.mean(), 1)}
display(tab_f)

nf = len(FOLDS)
M = np.zeros((nf, nf + 1))
for r, k in enumerate(FOLDS):
    M[r, :nf] = [1 if kk == k else 0 for kk in FOLDS]
    M[r, nf] = 2
fig, ax = plt.subplots(figsize=(1.5 * (nf + 1) + 1.5, 0.6 * nf + 1.8))
ax.imshow(M, cmap=ListedColormap(["#4c72b0", "#dd8452", "#bdbdbd"]), vmin=0, vmax=2, aspect="auto")
for r, k in enumerate(FOLDS):
    for j, kk in enumerate(FOLDS):
        ax.text(j, r, "VALIDA" if kk == k else "entrena", ha="center", va="center", color="white",
                fontsize=9, fontweight="bold" if kk == k else "normal")
    ax.text(nf, r, "no se toca", ha="center", va="center", fontsize=9)
n_por = {int(f.split()[1]) if f.startswith("fold") else 0: o for f, o in zip(tab_f.conjunto, tab_f.ojos) if f != "total"}
ax.set_xticks(range(nf + 1), [f"fold {k}\n({n_por[k]} ojos)" for k in FOLDS] + [f"test\n({n_por[0]} ojos)"])
ax.set_yticks(range(nf), [f"ronda {r + 1}" for r in range(nf)])
ax.set_title(f"Validación cruzada: en cada ronda se entrena con {nf - 1} fold{'s' if nf > 2 else ''} "
             "y se valida con el restante")
plt.tight_layout()
plt.savefig(OUT / "fig" / f"folds{TAG}.png", dpi=100)
plt.show()

### 1.3 Evaluación balanceada

Se **entrena con todos** los ojos del subset, pero **toda evaluación** (validación de cada fold,
OOF y test) se hace sobre un subconjunto con **la misma cantidad de sanos que de QC**: se conservan
todos los ojos de la clase minoritaria y se sortean (semilla 42) los de la otra. Así las matrices
de confusión son comparables entre sí y la accuracy no se infla por la clase mayoritaria. La
sensibilidad, la especificidad y el AUC no dependen de la proporción de clases, así que balancear
no las sesga. Los ojos que quedan afuera no se usan para medir.

In [ ]:
rng_ev = np.random.default_rng(SEED)


def sortear_balanceado(idx, y):
    """Índices de `idx` con igual cantidad por clase (toda la minoritaria + sorteo de la otra)."""
    pos, neg = idx[y == 1], idx[y == 0]
    n = min(len(pos), len(neg))
    return np.concatenate([rng_ev.choice(pos, n, replace=False), rng_ev.choice(neg, n, replace=False)])


EVAL_POOL = np.zeros(len(IDX_POOL), bool)            # máscara sobre IDX_POOL
for k in FOLDS:
    en_k = np.where(FOLD_POOL == k)[0]
    EVAL_POOL[sortear_balanceado(en_k, Y_POOL[en_k])] = True
TEST_BAL = np.zeros(len(IDX_TEST), bool)             # máscara sobre IDX_TEST
TEST_BAL[sortear_balanceado(np.arange(len(IDX_TEST)), Y_TEST)] = True

Y_POOL_EV, FOLD_EV = Y_POOL[EVAL_POOL], FOLD_POOL[EVAL_POOL]
Y_TEST_EV, PAC_TEST_EV = Y_TEST[TEST_BAL], PAC_TEST[TEST_BAL]

filas_ev = []
for k in FOLDS:
    m = FOLD_POOL == k
    filas_ev.append({"conjunto": f"validación fold {k}", "ojos totales": int(m.sum()),
                     "evaluados": int((m & EVAL_POOL).sum()),
                     "sanos evaluados": int(((Y_POOL == 0) & m & EVAL_POOL).sum()),
                     "QC evaluados": int(((Y_POOL == 1) & m & EVAL_POOL).sum())})
filas_ev.append({"conjunto": "OOF (todos los folds)", "ojos totales": len(IDX_POOL), "evaluados": int(EVAL_POOL.sum()),
                 "sanos evaluados": int((Y_POOL_EV == 0).sum()), "QC evaluados": int((Y_POOL_EV == 1).sum())})
filas_ev.append({"conjunto": "test", "ojos totales": len(IDX_TEST), "evaluados": int(TEST_BAL.sum()),
                 "sanos evaluados": int((Y_TEST_EV == 0).sum()), "QC evaluados": int((Y_TEST_EV == 1).sum())})
display(pd.DataFrame(filas_ev))
assert (Y_TEST_EV == 0).sum() == (Y_TEST_EV == 1).sum() and (Y_POOL_EV == 0).sum() == (Y_POOL_EV == 1).sum()
cerrar_etapa("Fase 1 — Datos, subset y folds")

## Fase 2 — Dataset sin letras T/N e inspección visual

Las letras **T** (temporal) y **N** (nasal) de las esquinas inferiores solo indican la lateralidad:
en OD aparece T–N y en OS N–T. No dicen nada del diagnóstico, pero el modelo puede usarlas como
atajo (y al espejar los OS la N queda invertida, "И", lo que delata el ojo).

Se genera una **copia** del dataset, `ORBSCAN_Dataset_modificado` (al lado de `ORBSCAN_Dataset`),
donde las dos esquinas inferiores se tapan con un **rectángulo blanco fijo**, igual para todas las
imágenes y para ambos ojos, así no deja ninguna información. **El original no se modifica.** Las
posiciones se midieron sobre las 5,628 imágenes: las letras llegan hasta ≈ 9.3 % del ancho (en
algunos estudios la N está un poco más corrida) y ocupan el último 10 % del alto; el rectángulo
cubre 10 % × 11 % para dejar margen. Al generar la copia se verifica **cada** imagen (que no quede
ninguna mancha negra del tamaño de una letra en las esquinas). La copia se genera una sola vez (las
siguientes corridas la reutilizan).

**Ojo:** los **números impresos** del Axial (curvaturas) y de la Paquimetría (espesores) **se
mantienen**. Son un posible atajo más fuerte; Grad-CAM (Fase 9) permite ver si el modelo los mira.

In [ ]:
iniciar_etapa("Fase 2 — Dataset sin T/N e inspección")
MARCA_X, MARCA_Y = 0.10, 0.11      # fracción del ancho / alto tapada en cada esquina inferior
MANCHA_MAX = 60                     # px: las letras miden 300-650 px; números/marcas de grilla, < 60


def mayor_mancha_oscura(im):
    """Tamaño (px) de la mayor mancha casi negra en las esquinas inferiores."""
    a = np.asarray(im.convert("RGB"))
    h, w = a.shape[:2]
    mx = 0
    for x0, x1 in [(0, int(0.2 * w)), (int(0.8 * w), w)]:
        lab, n = ndimage.label(a[int(0.8 * h):, x0:x1].max(2) < 60)
        if n:
            mx = max(mx, int(np.bincount(lab.ravel())[1:].max()))
    return mx


def cajas_marcas(w, h):
    bw, bh = int(math.ceil(MARCA_X * w)), int(math.ceil(MARCA_Y * h))
    return [(0, h - bh, bw, h), (w - bw, h - bh, w, h)]


def tapar_marcas(im):
    im = im.convert("RGB")
    dib = ImageDraw.Draw(im)
    for x0, y0, x1, y1 in cajas_marcas(*im.size):
        dib.rectangle([x0, y0, x1 - 1, y1 - 1], fill=(255, 255, 255))
    return im


def limpiar_marcas(src, dst):
    """Copia src -> dst tapando las letras T/N. Idempotente: si dst ya está completo, no hace nada."""
    pngs = sorted(src.glob("*/*/*.png"))
    info_f = dst / "_limpieza.json"
    if info_f.exists():
        info = json.loads(info_f.read_text(encoding="utf-8"))
        if info.get("n") == len(pngs) and info.get("cajas") == [MARCA_X, MARCA_Y] \
                and sum(1 for _ in dst.glob("*/*/*.png")) == len(pngs):
            desde_cache(info.get("segundos", 0))
            print(f"[cache] {dst} ya tiene las {len(pngs)} imágenes sin T/N")
            return dst
    t0, restos = time.time(), []
    info_f.unlink(missing_ok=True)
    for i, p in enumerate(pngs, start=1):
        q = dst / p.relative_to(src)
        q.parent.mkdir(parents=True, exist_ok=True)
        im = tapar_marcas(Image.open(p))
        if mayor_mancha_oscura(im) >= MANCHA_MAX:          # verificación de CADA imagen
            restos.append(str(p.relative_to(src)))
        im.save(q, compress_level=3)
        if i % 1000 == 0:
            print(f"  {i}/{len(pngs)}", flush=True)
    assert not restos, f"{len(restos)} imágenes con restos de letras, p.ej. {restos[:3]}: agrandar MARCA_X / MARCA_Y"
    seg = round(time.time() - t0, 1)
    info_f.write_text(json.dumps(dict(n=len(pngs), cajas=[MARCA_X, MARCA_Y], segundos=seg)), encoding="utf-8")
    print(f"{len(pngs)} imágenes sin T/N escritas y verificadas en {dst} ({seg:.0f} s)")
    return dst


if LIMPIAR_MARCAS:
    IMG_ROOT = limpiar_marcas(IMG_ROOT_ORIG, IMG_ROOT_ORIG.parent / "ORBSCAN_Dataset_modificado")
print("imágenes que usa el modelo:", IMG_ROOT)

muestra = np.random.default_rng(SEED).choice(len(MAN_FULL), size=min(40, len(MAN_FULL)), replace=False)
antes, despues = [], []
for i in muestra:
    for m in MAPAS:
        rel = MAN_FULL[f"rel_{m}"].iloc[i]
        antes.append(mayor_mancha_oscura(Image.open(IMG_ROOT_ORIG / rel)))
        despues.append(mayor_mancha_oscura(Image.open(IMG_ROOT / rel)))
print(f"mayor mancha negra en las esquinas inferiores ({len(antes)} imágenes) — "
      f"original: mín {min(antes)} px | usada por el modelo: máx {max(despues)} px")
if LIMPIAR_MARCAS:
    assert min(antes) > 200 and max(despues) < MANCHA_MAX, "las letras T/N no quedaron tapadas"

In [ ]:
r0 = MAN_FULL[MAN_FULL.label == 1].iloc[0]
fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
for j, m in enumerate(MAPAS):
    rel = r0[f"rel_{m}"]
    im0 = Image.open(IMG_ROOT_ORIG / rel).convert("RGB")
    axes[0, j].imshow(im0)
    for x0, y0, x1, y1 in cajas_marcas(*im0.size):
        axes[0, j].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="red", lw=1.5, ls="--"))
    axes[0, j].set_title(f"original | {r0.eye} | {m}\n(rojo = zona que se tapa)", fontsize=9)
    axes[1, j].imshow(Image.open(IMG_ROOT / rel))
    axes[1, j].set_title(f"ORBSCAN_Dataset_modificado | {m}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "fig" / f"marcas_TN_antes_despues{TAG}.png", dpi=90)
plt.show()

In [ ]:
def cargar_png(rel, eye):
    im = Image.open(IMG_ROOT / rel).convert("RGB")
    if ESPEJAR_OS and eye == "OS":
        im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    return im


ejemplos = []
for lab in (0, 1):
    ambos = MAN_FULL[MAN_FULL.label == lab].groupby("patient_code").filter(lambda d: set(d.eye) == {"OD", "OS"})
    pc = ambos.patient_code.iloc[0]
    ejemplos += [ambos[(ambos.patient_code == pc) & (ambos.eye == e)].iloc[0] for e in ("OD", "OS")]

fig, axes = plt.subplots(len(ejemplos), 4, figsize=(14, 3.6 * len(ejemplos)))
for i, r in enumerate(ejemplos):
    for j, m in enumerate(MAPAS):
        axes[i, j].imshow(cargar_png(r[f"rel_{m}"], r.eye))
        axes[i, j].set_title(f"{'QC' if r.label else 'sano'} | {r.eye} | {m}", fontsize=9)
        axes[i, j].axis("off")
fig.suptitle(f"Mapas tal como los ve el modelo (sin T/N = {LIMPIAR_MARCAS}, OS espejado = {ESPEJAR_OS})", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "fig" / f"inspeccion_mapas{TAG}.png", dpi=90)
plt.show()
cerrar_etapa("Fase 2 — Dataset sin T/N e inspección")

## Fase 3 — Infraestructura: métricas, umbral, matrices y registro de resultados

Todo experimento produce **probabilidades out-of-fold (OOF)** para el train y la **probabilidad
promedio de los modelos de cada fold** para el test. Con eso se calibra `tau` (solo OOF), se
evalúa el test sin tocarlo y se pueden fusionar mapas o modelos promediando probabilidades.
Todas las métricas se calculan sobre los subconjuntos **balanceados** de la Fase 1.3.

* **Matrices de confusión**: filas = clase real, columnas = clase predicha. Cada celda muestra
  `k/n` = cuántos ojos de esa clase real cayeron en esa predicción, sobre el total de la clase
  (p. ej. `14/15`). El color es esa fracción. El **pie** indica cuántos ojos se evaluaron por
  clase, cuántas imágenes por ojo usa el modelo y el `tau` aplicado.
* **Resaltado del elegido**: fila en verde, celda del criterio en verde oscuro y encabezado de esa
  columna en verde, más un renglón "✅ ELEGIDO" con el valor del criterio.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + z ** 2 / n
    centro = (p + z ** 2 / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return centro - half, centro + half


def elegir_tau(y, p, objetivo=OBJ_SENS):
    """Mayor tau (-> mayor especificidad) con sensibilidad >= objetivo. Regla: QC si p >= tau."""
    pos = np.sort(np.asarray(p, float)[np.asarray(y) == 1])[::-1]
    if len(pos) == 0:
        return 0.5
    return float(pos[int(math.ceil(objetivo * len(pos))) - 1])


def esp_a_sens(y, p, objetivo=OBJ_SENS):
    """Especificidad máxima alcanzable con sensibilidad >= objetivo (sin umbral fijado)."""
    if len(np.unique(y)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(y, p, drop_intermediate=False)
    ok = tpr >= objetivo - 1e-12
    return float(1 - fpr[ok].min()) if ok.any() else float("nan")


def bce(y, p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def metricas_bin(y, p, tau):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, float)
    pred = (p >= tau).astype(int)
    tp = int(((y == 1) & (pred == 1)).sum()); fn = int(((y == 1) & (pred == 0)).sum())
    tn = int(((y == 0) & (pred == 0)).sum()); fp = int(((y == 0) & (pred == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else float("nan")
    esp = tn / (tn + fp) if tn + fp else float("nan")
    prec = tp / (tp + fp) if tp + fp else 0.0
    f2 = 5 * prec * sens / (4 * prec + sens) if (4 * prec + sens) > 0 else 0.0
    dos = len(np.unique(y)) == 2
    return dict(tau=float(tau), sens=sens, sens_ic=list(wilson(tp, tp + fn)), esp=esp,
                esp_ic=list(wilson(tn, tn + fp)), bal_acc=(sens + esp) / 2, f2=float(f2),
                tp=tp, fn=fn, tn=tn, fp=fp, n=len(y), n0=int((y == 0).sum()), n1=int((y == 1).sum()),
                auc=float(roc_auc_score(y, p)) if dos else float("nan"),
                ap=float(average_precision_score(y, p)) if dos else float("nan"),
                brier=float(brier_score_loss(y, p)), esp_a_sens=esp_a_sens(y, p))


def boot_ci(y, p, pacientes, tau, B=None, seed=SEED):
    """IC 95 % por bootstrap re-muestreando PACIENTES (los 2 ojos de un paciente van juntos)."""
    B = B or N_BOOT
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y), np.asarray(p, float)
    ids = np.unique(pacientes)
    grupos = {i: np.where(pacientes == i)[0] for i in ids}
    acc = {"auc": [], "sens": [], "esp": []}
    for _ in range(B):
        sel = np.concatenate([grupos[i] for i in rng.choice(ids, len(ids), replace=True)])
        ys, ps = y[sel], p[sel]
        if len(np.unique(ys)) < 2:
            continue
        pred = ps >= tau
        acc["auc"].append(roc_auc_score(ys, ps))
        acc["sens"].append(pred[ys == 1].mean())
        acc["esp"].append((~pred[ys == 0]).mean())
    return {k: ([float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] if v else [float("nan")] * 2)
            for k, v in acc.items()}

In [ ]:
RESULTADOS = {}


def ruta_res(exp_id):
    return OUT / "res" / f"{exp_id}{TAG}.json"


def _claves(idx):
    return man["eye_key"].to_numpy(dtype=str)[idx].tolist()


def armar_res(exp_id, familia, oof, p_test, extra=None, segundos=0.0):
    """Evalúa (sobre los subconjuntos balanceados) y guarda un experimento."""
    oof, p_test = np.asarray(oof, float), np.asarray(p_test, float)
    assert not np.isnan(oof).any() and len(oof) == len(IDX_POOL) and len(p_test) == len(IDX_TEST)
    o_ev, t_ev = oof[EVAL_POOL], p_test[TEST_BAL]
    auc_folds = [float(roc_auc_score(Y_POOL_EV[FOLD_EV == k], o_ev[FOLD_EV == k])) for k in FOLDS]
    tau = elegir_tau(Y_POOL_EV, o_ev)
    res = dict(id=exp_id, familia=familia, extra=extra or {}, segundos=round(float(segundos), 1),
               auc_folds=auc_folds, auc_cv_media=float(np.mean(auc_folds)), auc_cv_sd=float(np.std(auc_folds)),
               oof=metricas_bin(Y_POOL_EV, o_ev, tau), test=metricas_bin(Y_TEST_EV, t_ev, tau),
               test_boot=boot_ci(Y_TEST_EV, t_ev, PAC_TEST_EV, tau),
               oof_p=np.round(oof, 6).tolist(), test_p=np.round(p_test, 6).tolist(),
               claves_pool=_claves(IDX_POOL), claves_test=_claves(IDX_TEST),
               claves_ev=[_claves(IDX_POOL[EVAL_POOL]), _claves(IDX_TEST[TEST_BAL])])
    ruta_res(exp_id).write_text(json.dumps(res, indent=1), encoding="utf-8")
    return res


def cargar_res(exp_id):
    f = ruta_res(exp_id)
    if not f.exists():
        return None
    r = json.loads(f.read_text(encoding="utf-8"))
    if r.get("claves_pool") != _claves(IDX_POOL) or r.get("claves_test") != _claves(IDX_TEST):
        return None
    desde_cache(r.get("segundos", 0))
    if r.get("claves_ev") != [_claves(IDX_POOL[EVAL_POOL]), _claves(IDX_TEST[TEST_BAL])]:
        r = armar_res(exp_id, r["familia"], r["oof_p"], r["test_p"], r["extra"], r.get("segundos", 0))
    print(f"[cache] {exp_id}")
    return r


def fusionar(exp_id, ids, familia, extra=None):
    """Fusión tardía: promedio de probabilidades (OOF y test) de experimentos ya corridos."""
    rs = [RESULTADOS[i] for i in ids]
    return armar_res(exp_id, familia, np.mean([r["oof_p"] for r in rs], 0),
                     np.mean([r["test_p"] for r in rs], 0), dict(extra or {}, componentes=ids))


def fila(r):
    o, t, b, e = r["oof"], r["test"], r["test_boot"], r["extra"]
    return dict(ID=r["id"], familia=r["familia"], backbone=e.get("backbone", "-"), entrada=e.get("entrada", "-"),
                AUC_cv=r["auc_cv_media"], AUC_cv_sd=r["auc_cv_sd"], AUC_oof=o["auc"], tau=o["tau"],
                Sens_oof=o["sens"], Esp_oof=o["esp"],
                AUC_test=t["auc"], AUC_ic=f"[{b['auc'][0]:.3f}-{b['auc'][1]:.3f}]",
                Sens_test=t["sens"], Sens_ic_wilson=f"[{t['sens_ic'][0]:.3f}-{t['sens_ic'][1]:.3f}]",
                Esp_test=t["esp"], Esp_ic_boot=f"[{b['esp'][0]:.3f}-{b['esp'][1]:.3f}]",
                Esp_a_sens95_test=t["esp_a_sens"], AP_test=t["ap"], F2_test=t["f2"], BalAcc_test=t["bal_acc"],
                Brier_test=t["brier"], FN_test=t["fn"], FP_test=t["fp"], n_test=f"{t['n0']}+{t['n1']}",
                seg=r.get("segundos", 0))


def tabla(ids):
    return pd.DataFrame([fila(RESULTADOS[i]) for i in ids if i in RESULTADOS])


def cv_sklearn(exp_id, familia, X, make_clf, extra=None):
    """CV con los folds oficiales. X: matriz (ojos x features) o función fold -> matriz."""
    r = cargar_res(exp_id)
    if r:
        return r
    t0 = time.time()
    oof, p_test = np.full(len(IDX_POOL), np.nan), []
    for k in FOLDS:
        Xk = X(k) if callable(X) else X
        tr, va = IDX_POOL[FOLD_POOL != k], FOLD_POOL == k
        clf = make_clf().fit(Xk[tr], Y[tr])
        oof[va] = clf.predict_proba(Xk[IDX_POOL[va]])[:, 1]
        p_test.append(clf.predict_proba(Xk[IDX_TEST])[:, 1])
    return armar_res(exp_id, familia, oof, np.mean(p_test, 0), extra, segundos=time.time() - t0)

In [ ]:
VERDE, VERDE_OSC = "#d9f2d9", "#1a9850"
COLS_VAL = ["ID", "entrada", "AUC_cv", "AUC_cv_sd", "AUC_oof", "tau", "Sens_oof", "Esp_oof"]
COLS_TEST = ["ID", "backbone", "entrada", "AUC_oof", "Esp_oof", "tau", "AUC_test", "AUC_ic", "Sens_test",
             "Sens_ic_wilson", "Esp_test", "Esp_ic_boot", "BalAcc_test", "n_test"]
COLS_FINAL = ["ID", "familia", "backbone", "entrada", "AUC_oof", "Esp_oof", "tau", "AUC_test", "AUC_ic",
              "Sens_test", "Sens_ic_wilson", "Esp_test", "Esp_ic_boot", "Esp_a_sens95_test", "AP_test",
              "F2_test", "Brier_test", "FN_test", "FP_test", "n_test"]


def y_p(r, cual):
    """Etiquetas y probabilidades del subconjunto balanceado: cual = 'oof' (validación) o 'test'."""
    if cual == "oof":
        return Y_POOL_EV, np.asarray(r["oof_p"])[EVAL_POOL]
    return Y_TEST_EV, np.asarray(r["test_p"])[TEST_BAL]


def plot_cm(ax, y, p, tau, titulo, fuente, resaltado=False):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= tau).astype(int)
    cm = np.array([[int(((y == i) & (pred == j)).sum()) for j in (0, 1)] for i in (0, 1)])
    n_cl = cm.sum(1)
    frac = cm / np.maximum(n_cl[:, None], 1)
    ax.imshow(frac, cmap="Blues", vmin=0, vmax=1)
    for i in (0, 1):
        for j in (0, 1):
            ax.text(j, i, f"{cm[i, j]}/{n_cl[i]}", ha="center", va="center", fontsize=13, fontweight="bold",
                    color="white" if frac[i, j] > 0.55 else "black")
    ax.set_xticks([0, 1], ["Sano", "QC"])
    ax.set_yticks([0, 1], ["Sano", "QC"])
    ax.set_xlabel("predicha")
    ax.set_ylabel("real")
    ax.set_title(("★ ELEGIDO\n" if resaltado else "") + titulo, fontsize=9,
                 color=VERDE_OSC if resaltado else "black", fontweight="bold" if resaltado else "normal")
    ax.text(0.5, -0.30, f"n = {len(y)} ojos ({n_cl[0]} Sano / {n_cl[1]} QC)\n{fuente}\ntau = {tau:.3f}",
            transform=ax.transAxes, ha="center", va="top", fontsize=7.5)
    if resaltado:
        for s in ax.spines.values():
            s.set_edgecolor(VERDE_OSC)
            s.set_linewidth(4)


def cm_res(ax, eid, cual, titulo=None, resaltado=False):
    r = RESULTADOS[eid]
    y, p = y_p(r, cual)
    m = r["oof"] if cual == "oof" else r["test"]
    tit = titulo or eid
    tit += f"\nAUC={m['auc']:.3f} · sens={m['sens']:.2f} · esp={m['esp']:.2f}"
    fuente = ("validación (OOF)" if cual == "oof" else "test") + " · " + r["extra"].get("entrada", "")
    plot_cm(ax, y, p, r["oof"]["tau"], tit, fuente, resaltado)


def fila_cms(ids, cual, titulo, archivo, mejor=None):
    fig, axes = plt.subplots(1, len(ids), figsize=(3.5 * len(ids), 4.6), squeeze=False)
    for ax, eid in zip(axes[0], ids):
        cm_res(ax, eid, cual, resaltado=(eid == mejor))
    fig.suptitle(titulo, fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / "fig" / f"{archivo}{TAG}.png", dpi=100)
    plt.show()


def elegir(df, excluir=("TAB_LR",)):
    d = df[~df.ID.isin(excluir)].dropna(subset=[CRITERIO])
    d = d.assign(_k=d[CRITERIO].round(3), _d=d[DESEMPATE].fillna(-1))
    return d.sort_values(["_k", "_d"], ascending=False).iloc[0]["ID"]


def resaltar(df, cols=None, excluir=("TAB_LR",), titulo="", orden=False):
    """Muestra la tabla con el elegido resaltado (fila verde + celda del criterio) y devuelve su ID."""
    ganador = elegir(df, excluir)
    df = df.copy()
    if orden:
        df = df.assign(_k=df[CRITERIO].round(3), _d=df[DESEMPATE].fillna(-1)) \
               .sort_values(["_k", "_d"], ascending=False).drop(columns=["_k", "_d"])
    df = df.reset_index(drop=True)
    cols = [c for c in (cols or list(df.columns)) if c in df.columns]
    if CRITERIO not in cols:
        cols.insert(1, CRITERIO)
    i_g = int(df.index[df.ID == ganador][0])
    ref = set(df.index[df.ID.isin(excluir)])

    def estilo_fila(row):
        if row.name == i_g:
            return [f"background-color:{VERDE}"] * len(row)
        if row.name in ref:
            return ["color:#888888;font-style:italic"] * len(row)
        return [""] * len(row)

    j = cols.index(CRITERIO)
    sty = (df[cols].style.apply(estilo_fila, axis=1)
           .set_properties(subset=pd.IndexSlice[[i_g], [CRITERIO]],
                           **{"background-color": VERDE_OSC, "color": "white", "font-weight": "bold"})
           .format(precision=4, na_rep="-")
           .set_table_styles([{"selector": f"th.col_heading.col{j}",
                               "props": f"background-color:{VERDE_OSC};color:white"}], overwrite=False)
           .hide(axis="index"))
    g = df.loc[i_g]
    nota_ref = f" Filas en gris = referencia (no compiten): {', '.join(df.ID[list(ref)])}." if ref else ""
    display(Markdown(f"✅ **ELEGIDO{' — ' + titulo if titulo else ''}: `{ganador}`** · criterio "
                     f"**{CRITERIO} = {g[CRITERIO]:.4f}** (desempate {DESEMPATE} = {g[DESEMPATE]:.4f}). "
                     f"Fila en verde; la columna del criterio tiene el encabezado verde.{nota_ref}"))
    display(sty)
    return ganador

## Fase 4 — Baseline tabular (variables clínicas): la vara de comparación

**Qué hace.** Toma solo la planilla del CSV —13 números por ojo: Kmax y su eje, paquimetría
central y mínima, ubicación del punto más delgado, astigmatismo y su eje, asfericidades anterior y
posterior, y edad— y entrena una **regresión logística**: una fórmula que le da un peso a cada
número y devuelve una probabilidad de QC. **No mira ninguna imagen.**

**Ejemplo.** Un ojo con Kmax = 58 D y paquimetría mínima = 430 µm sale con P(QC) ≈ 0.97; uno con
Kmax = 43 D y 540 µm, con P(QC) ≈ 0.03.

**Para qué sirve (analogía).** Antes de pagarle a un especialista (la red neuronal) para que lea el
estudio, conviene ver cuánto se acierta solo leyendo el informe numérico que ya imprime el equipo.
Si la red **no le gana** a la planilla, las imágenes no agregan información y el proyecto se tiene
que justificar por otro lado (automatizar, independencia del equipo, explicabilidad). Si **le
gana**, hay un argumento cuantitativo de que las imágenes aportan.

**Detalles.** Variables estandarizadas, regularización L2, `class_weight="balanced"`, mismos folds,
mismo `tau` y misma evaluación balanceada que el resto. **No** se usa `gender` (EDA §7). Es una vara
**exigente**: el Orbscan calcula esos números a partir de los mismos mapas. **No compite** por el
modelo elegido: aparece en gris como referencia en las tablas.

In [ ]:
iniciar_etapa("Fase 4 — Baseline tabular")
X_TAB = man[VARS_TAB].values.astype(float)


def make_lr_tab():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C_LR_TAB, class_weight="balanced", max_iter=5000))


RESULTADOS["TAB_LR"] = cv_sklearn("TAB_LR", "baseline tabular (referencia)", X_TAB, make_lr_tab,
                                  dict(backbone="-", mapa="clínicas", clf="LogReg",
                                       entrada="13 variables clínicas (sin imágenes)"))
display(tabla(["TAB_LR"])[COLS_TEST].round(4))

coef = make_lr_tab().fit(X_TAB[IDX_POOL], Y_POOL)[-1].coef_[0]
display(pd.Series(coef, index=VARS_TAB, name="peso estandarizado (ajuste sobre todo el train)")
        .sort_values(key=np.abs, ascending=False).round(3).to_frame())
fila_cms(["TAB_LR"], "test", "Baseline tabular — test", "cm_tabular")
cerrar_etapa("Fase 4 — Baseline tabular")

## Fase 5 — Análisis de backbones

Se compara el **transfer learning** de tres redes preentrenadas en ImageNet, de menor a mayor
tamaño: **MobileNetV2** (3.5 M parámetros, 224 px), **ResNet50** (25.6 M, 224 px) y
**EfficientNet-B7** (66 M, 600 px).

Para cada red y cada mapa:
1. El backbone queda **congelado** y se calcula una vez su vector de características (salida del
   *global average pooling*) para cada ojo. Se cachea en `emb/`.
2. Se entrena **solo la última capa** (`Dropout(0.3) → Linear(1)`) con `BCEWithLogits(pos_weight)`,
   Adam y **early stopping por AUC de validación** (balanceada) del fold. Como el backbone está
   congelado y en esta fase no hay aumento de datos, entrenar sobre los vectores cacheados es
   matemáticamente lo mismo que pasar las imágenes por la red, pero mucho más rápido (es lo que hace
   viable a B7).
3. Se entrena un modelo por fold: su predicción de validación arma el OOF y el test recibe el
   promedio de los 5.
4. **Fusión**: se promedian las probabilidades de los 4 mapas.

Por cada red se muestra el **avance del entrenamiento** (pérdida y AUC de validación por época y por
fold) y la **matriz de confusión de validación** (OOF balanceado, con el `tau` de sensibilidad ≥
0.95). En **5.4** están los resultados en **test** y la elección del backbone que pasa al
fine-tuning.

In [ ]:
_CACHE_IMG = {}


def imagenes_mapa(mapa, size):
    """Lista (alineada con man) de imágenes pre-redimensionadas a size*1.07 (OS espejado). Caché en RAM por mapa."""
    pre = int(round(size * 1.07))
    if any(k[0] != mapa for k in _CACHE_IMG):
        _CACHE_IMG.clear()
    if (mapa, pre) not in _CACHE_IMG:
        _CACHE_IMG[(mapa, pre)] = [cargar_png(rel, eye).resize((pre, pre), Image.BILINEAR)
                                   for rel, eye in zip(man[f"rel_{mapa}"], man["eye"])]
    return _CACHE_IMG[(mapa, pre)]


def ops_aug(size):
    """Aumento de datos (solo train del fine-tuning): encuadre/zoom leve, rotación ±5°, traslación 3 %."""
    return [T.RandomResizedCrop(size, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
            T.RandomAffine(degrees=5, translate=(0.03, 0.03), fill=255)]


def build_tf(size, aug, mean, std):
    ops = ops_aug(size) if aug else [T.CenterCrop(size)]
    return T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])


class MapaDS(Dataset):
    def __init__(self, imgs, idx, tf):
        self.imgs, self.idx, self.tf = imgs, np.asarray(idx), tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return self.tf(self.imgs[j]), np.float32(Y[j])


class Clf(nn.Module):
    def __init__(self, bb_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(bb_name, pretrained=pretrained, num_classes=0)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.num_features, 1))
        cfg = self.backbone.pretrained_cfg
        self.mean, self.std = tuple(cfg["mean"]), tuple(cfg["std"])

    def train(self, mode=True):
        super().train(mode)
        if mode:
            self.backbone.eval()      # BN/dropout del backbone siempre en eval (batches chicos)
        return self

    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(1)


def bs_eval(size):
    return 16 if size > 400 else 64


@torch.no_grad()
def predecir(model, ds, size):
    model.eval()
    out = []
    for x, _ in DataLoader(ds, batch_size=bs_eval(size), shuffle=False, num_workers=NUM_WORKERS):
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out.append(torch.sigmoid(model(x.to(DEVICE)).float()).cpu().numpy())
    return np.concatenate(out)


@torch.no_grad()
def emb_modelo(model, imgs, size):
    """Vector de características (salida del pooling del backbone) de cada ojo de `man`."""
    model.eval()
    ds = MapaDS(imgs, np.arange(len(man)), build_tf(size, False, model.mean, model.std))
    feats = []
    for x, _ in DataLoader(ds, batch_size=bs_eval(size), shuffle=False, num_workers=NUM_WORKERS):
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            feats.append(model.backbone(x.to(DEVICE)).float().cpu().numpy())
    X = np.concatenate(feats).astype(np.float32)
    assert np.isfinite(X).all()
    return X


def _leer_npz(f):
    if f.exists():
        d = np.load(f, allow_pickle=False)
        if d["eye_key"].tolist() == man["eye_key"].to_numpy(dtype=str).tolist():
            desde_cache(float(d["segundos"]))
            return d["X"]
    return None


def _guardar_npz(f, X, t0):
    np.savez_compressed(f, X=X, eye_key=man["eye_key"].to_numpy(dtype=str), segundos=time.time() - t0)


def liberar(*objs):
    del objs
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def extraer_emb(bb, size, mapa):
    f = OUT / "emb" / f"emb_{CORTO[bb]}_{mapa}{TAG}.npz"
    X = _leer_npz(f)
    if X is not None:
        return X
    t0 = time.time()
    model = Clf(bb).to(DEVICE)
    X = emb_modelo(model, imagenes_mapa(mapa, size), size)
    _guardar_npz(f, X, t0)
    print(f"  embeddings {CORTO[bb]} {mapa}: {X.shape} ({time.time() - t0:.0f} s)")
    liberar(model)
    return X


def ruta_ckpt(exp_id, k):
    return OUT / "ckpt" / f"{exp_id}_f{k}{TAG}.pth"


def cargar_clf(bb, exp_id, k):
    """Clf con pesos de ImageNet + lo entrenado en `exp_id` (fold k): la cabeza y, si hubo FT, las capas liberadas."""
    m = Clf(bb).to(DEVICE)
    sd = torch.load(ruta_ckpt(exp_id, k), map_location=DEVICE)
    inc = m.load_state_dict({n: v.float() for n, v in sd.items()}, strict=False)
    assert not inc.unexpected_keys, inc.unexpected_keys
    return m.eval()


def entrenar_cabeza_cv(exp_id, bb, mapa):
    """Entrena solo la última capa sobre los embeddings congelados, en CV con los folds oficiales."""
    r = cargar_res(exp_id)
    if r and all(ruta_ckpt(exp_id, k).exists() for k in FOLDS):
        return r
    cfg, t0 = HEAD_CFG, time.time()
    X_all = EMB[(bb, mapa)]
    yt = torch.tensor(Y, dtype=torch.float32, device=DEVICE)
    oof, p_test, hist_all = np.full(len(IDX_POOL), np.nan), [], {}
    for k in FOLDS:
        set_seed(SEED + k)
        tr_idx, va_mask = IDX_POOL[FOLD_POOL != k], FOLD_POOL == k
        va_idx, va_ev = IDX_POOL[va_mask], IDX_POOL[va_mask & EVAL_POOL]
        t_va, t_va_ev, t_te = (torch.as_tensor(a, device=DEVICE) for a in (va_idx, va_ev, IDX_TEST))
        mu, sd = X_all[tr_idx].mean(0), X_all[tr_idx].std(0) + 1e-6     # estandarización con el train del fold
        Xs = torch.tensor((X_all - mu) / sd, dtype=torch.float32, device=DEVICE)
        head = nn.Sequential(nn.Dropout(0.3), nn.Linear(X_all.shape[1], 1)).to(DEVICE)
        opt = torch.optim.Adam(head.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
        n1 = int(Y[tr_idx].sum())
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(tr_idx) - n1) / max(n1, 1), device=DEVICE))
        gen = torch.Generator().manual_seed(SEED + k)
        best_auc, best_state, best_ep, wait, hist = -1.0, None, 0, 0, []
        for ep in range(1, cfg["epochs"] + 1):
            head.train()
            perm = torch.as_tensor(tr_idx)[torch.randperm(len(tr_idx), generator=gen)]
            ep_loss = 0.0
            for s in range(0, len(perm), cfg["batch"]):
                b = perm[s:s + cfg["batch"]].to(DEVICE)
                opt.zero_grad(set_to_none=True)
                loss = crit(head(Xs[b]).squeeze(1), yt[b])
                loss.backward()
                opt.step()
                ep_loss += loss.item() * len(b)
            head.eval()
            with torch.no_grad():
                p_va = torch.sigmoid(head(Xs[t_va_ev]).squeeze(1)).cpu().numpy()
            auc = float(roc_auc_score(Y[va_ev], p_va))
            hist.append(dict(epoch=ep, train_loss=round(ep_loss / len(perm), 4),
                             val_loss=round(bce(Y[va_ev], p_va), 4), val_auc=round(auc, 4)))
            if best_state is None or auc > best_auc + 1e-4:
                best_auc, best_ep, wait = auc, ep, 0
                best_state = {kk: v.detach().clone() for kk, v in head.state_dict().items()}
            else:
                wait += 1
                if wait >= cfg["patience"]:
                    break
        head.load_state_dict(best_state)
        head.eval()
        with torch.no_grad():
            oof[va_mask] = torch.sigmoid(head(Xs[t_va]).squeeze(1)).cpu().numpy()
            p_test.append(torch.sigmoid(head(Xs[t_te]).squeeze(1)).cpu().numpy())
        # cabeza expresada sobre las features crudas -> se carga directo en Clf para el FT y Grad-CAM
        W, b0 = best_state["1.weight"].float().cpu(), best_state["1.bias"].float().cpu()
        mu_t, sd_t = torch.tensor(mu, dtype=torch.float32), torch.tensor(sd, dtype=torch.float32)
        torch.save({"head.1.weight": W / sd_t, "head.1.bias": b0 - (W * (mu_t / sd_t)).sum(1)}, ruta_ckpt(exp_id, k))
        hist_all[k] = hist
        print(f"  {exp_id} fold {k}: {len(hist)} épocas, mejor época = {best_ep}, AUC val = {best_auc:.4f}", flush=True)
    return armar_res(exp_id, "TL última capa", oof, np.mean(p_test, 0),
                     dict(backbone=bb, mapa=mapa, entrada=f"1 imagen por ojo ({mapa})", historia=hist_all),
                     segundos=time.time() - t0)


def graficar_historia(ids, titulo, archivo):
    fig, axes = plt.subplots(2, len(ids), figsize=(4.3 * len(ids), 6.6), squeeze=False)
    colores = plt.cm.tab10(np.arange(10))
    for j, eid in enumerate(ids):
        h = RESULTADOS[eid]["extra"]["historia"]
        mapa = RESULTADOS[eid]["extra"]["mapa"]
        for ci, (k, hk) in enumerate(h.items()):
            ep = [e["epoch"] for e in hk]
            axes[0, j].plot(ep, [e["train_loss"] for e in hk], color=colores[ci], lw=1.2, label=f"fold {k}")
            axes[0, j].plot(ep, [e["val_loss"] for e in hk], color=colores[ci], lw=1.2, ls="--")
            aucs = [e["val_auc"] for e in hk]
            axes[1, j].plot(ep, aucs, color=colores[ci], lw=1.2, label=f"fold {k}")
            ib = int(np.argmax(aucs))
            axes[1, j].scatter(ep[ib], aucs[ib], color=colores[ci], s=28, zorder=5)
        axes[0, j].set_title(f"{mapa}: pérdida (— train, - - val)", fontsize=9)
        axes[1, j].set_title(f"{mapa}: AUC de validación (● = época elegida)", fontsize=9)
        axes[1, j].set_xlabel("época")
        axes[1, j].set_ylim(0.4, 1.01)
        for a in axes[:, j]:
            a.xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[1, 0].legend(fontsize=7)
    fig.suptitle(titulo, fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / "fig" / f"{archivo}{TAG}.png", dpi=100)
    plt.show()

In [ ]:
iniciar_etapa("Fase 5 — Análisis de backbones")
EMB, IDS_CAB = {}, []
for i_bb, (bb, size) in enumerate(BACKBONES.items(), start=1):
    c = CORTO[bb]
    display(Markdown(f"### 5.{i_bb} `{bb}` — {size} px"))
    with subetapa(f"Fase 5 · {c}", "Fase 5 — Análisis de backbones"):
        ids_bb = []
        for mapa in MAPAS:
            EMB[(bb, mapa)] = extraer_emb(bb, size, mapa)
            eid = f"CAB_{c}_{mapa}"
            RESULTADOS[eid] = entrenar_cabeza_cv(eid, bb, mapa)
            ids_bb.append(eid)
        eid = f"CAB_{c}_fusion"
        RESULTADOS[eid] = fusionar(eid, ids_bb, "TL última capa, fusión 4 mapas",
                                   dict(backbone=bb, mapa="fusión",
                                        entrada="4 imágenes por ojo (promedio de los 4 mapas)"))
        ids_bb.append(eid)
    IDS_CAB += ids_bb
    graficar_historia(ids_bb[:-1], f"{c}: avance del entrenamiento de la última capa (por fold)", f"historia_cab_{c}")
    mejor_bb = resaltar(tabla(ids_bb), COLS_VAL, titulo=f"mejor de {c} (validación)")
    fila_cms(ids_bb, "oof", f"{c}: matrices de confusión de VALIDACIÓN (OOF balanceado)", f"cm_val_cab_{c}",
             mejor=mejor_bb)

### 5.4 Resultados en test y elección del backbone

El test (balanceado) se predice con el promedio de los 5 modelos de cada experimento, aplicando el
`tau` calibrado en validación. **La elección se hace con el criterio de validación** (`AUC_oof`);
las columnas de test son solo informativas. El **backbone del modelo elegido** es el que pasa al
fine-tuning (Fase 6).

In [ ]:
MEJOR_CAB = resaltar(tabla(IDS_CAB), COLS_TEST, titulo="transfer learning (última capa)")
BEST_BB = RESULTADOS[MEJOR_CAB]["extra"]["backbone"]
BEST_SIZE = BACKBONES[BEST_BB]
c = CORTO[BEST_BB]

cols_cm = MAPAS + ["fusion"]
fig, axes = plt.subplots(len(BACKBONES), len(cols_cm), figsize=(3.5 * len(cols_cm), 4.7 * len(BACKBONES)),
                         squeeze=False)
for i, bb in enumerate(BACKBONES):
    for j, mapa in enumerate(cols_cm):
        eid = f"CAB_{CORTO[bb]}_{mapa}"
        cm_res(axes[i, j], eid, "test", resaltado=(eid == MEJOR_CAB))
fig.suptitle("Matrices de confusión en TEST (balanceado) — todos los modelos de transfer learning", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "fig" / f"cm_test_todos_cab{TAG}.png", dpi=90)
plt.show()
print(f"→ backbone que pasa al fine-tuning: {BEST_BB} ({BEST_SIZE} px), porque el modelo elegido es {MEJOR_CAB}")
cerrar_etapa("Fase 5 — Análisis de backbones")

## Fase 6 — Fine-tuning del mejor backbone (15 % final de las capas), con aumento de datos

### ¿Se hace aumento de datos? Sí: acá
El aumento de datos (*data augmentation*) genera en cada época una versión levemente distinta de
cada imagen de entrenamiento, para que la red no memorice detalles irrelevantes. Se usa **solo en el
fine-tuning** (en la Fase 5 el backbone está congelado y se entrena sobre vectores fijos) y **solo en
train**: validación y test van sin aumento.

| Transformación | Valor | Para qué |
|---|---|---|
| `RandomResizedCrop` | 85-100 % del área, relación 0.95-1.05 | pequeñas diferencias de encuadre y zoom |
| `RandomAffine` (rotación) | ±5° | pequeñas diferencias de alineación |
| `RandomAffine` (traslación) | ±3 % | descentrado leve |

**No** se usa: volteo vertical (la asimetría inferior/superior es un signo de QC), rotaciones
grandes (cambian qué zona queda abajo) ni cambios de color o brillo (el color **es** el valor medido
del mapa). Tampoco volteo horizontal: los OS ya se espejan para que todos los ojos tengan la misma
orientación.

### Fine-tuning
Se parte de la última capa entrenada en la Fase 5 (mismo fold) y se **descongela el 15 % final de
las capas** del backbone ganador. "Capa" = módulo con parámetros propios (convoluciones,
BatchNorm, lineales), contadas en el orden en que se aplican; se liberan las últimas
`ceil(0.15 · N)`. AdamW (LR backbone 1e-4, cabeza 3e-4, weight decay 1e-2), cosine, AMP y early
stopping por AUC de validación balanceada. "Antes" (Fase 5) y "después" se evalúan con la misma
función, los mismos folds y el mismo test.

In [ ]:
iniciar_etapa("Fase 6 — Fine-tuning")
torch.manual_seed(SEED)
j_ej = int(IDX_POOL[Y_POOL == 1][0])
aug_vis = T.Compose(ops_aug(BEST_SIZE))
fig, axes = plt.subplots(len(MAPAS), 5, figsize=(13, 2.7 * len(MAPAS)))
for i, mapa in enumerate(MAPAS):
    pre = int(round(BEST_SIZE * 1.07))
    im = cargar_png(man[f"rel_{mapa}"].iloc[j_ej], man.eye.iloc[j_ej]).resize((pre, pre), Image.BILINEAR)
    axes[i, 0].imshow(T.CenterCrop(BEST_SIZE)(im))
    axes[i, 0].set_title(f"{mapa}: original (val/test)", fontsize=9)
    for jj in range(1, 5):
        axes[i, jj].imshow(aug_vis(im))
        axes[i, jj].set_title(f"aumentada #{jj} (solo train)", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle(f"Ejemplos de aumento de datos ({BEST_SIZE} px)", fontsize=12)
plt.tight_layout()
plt.savefig(OUT / "fig" / f"aumento_datos{TAG}.png", dpi=80)
plt.show()

In [ ]:
def capas_con_parametros(mod):
    """Módulos con parámetros propios, en orden de registro (= orden de forward en estas redes)."""
    return [(n, m) for n, m in mod.named_modules() if n and next(m.parameters(recurse=False), None) is not None]


def descongelar_pct(model, pct):
    for p in model.parameters():
        p.requires_grad = False
    capas = capas_con_parametros(model.backbone)
    n = max(1, math.ceil(pct * len(capas)))
    for _, m in capas[-n:]:
        for p in m.parameters(recurse=False):
            p.requires_grad = True
    for p in model.head.parameters():
        p.requires_grad = True
    n_lib = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
    n_tot = sum(p.numel() for p in model.backbone.parameters())
    return {"capas con parámetros": len(capas), "capas liberadas": n, "primera liberada": capas[-n][0],
            "parámetros liberados": n_lib, "% parámetros": round(100 * n_lib / n_tot, 1)}


filas_u = []
for bb in BACKBONES:
    m_ = Clf(bb, pretrained=False)
    filas_u.append({"backbone": bb, **descongelar_pct(m_, FT_CFG["pct_capas"])})
    del m_
display(Markdown(f"**Qué significa descongelar el {FT_CFG['pct_capas']:.0%} final de las capas en cada red** "
                 f"(se aplica solo al backbone elegido, `{BEST_BB}`):"))
display(pd.DataFrame(filas_u))


def entrenar_ft_cv(exp_id, bb, size, mapa, init_id):
    r = cargar_res(exp_id)
    if r and all(ruta_ckpt(exp_id, k).exists() for k in FOLDS):
        return r
    cfg = dict(FT_CFG, batch=FT_BATCH.get(CORTO[bb], FT_CFG["batch"]))
    imgs = imagenes_mapa(mapa, size)
    oof, p_test, hist_all, t0 = np.full(len(IDX_POOL), np.nan), [], {}, time.time()
    for k in FOLDS:
        set_seed(SEED + k)
        model = cargar_clf(bb, init_id, k)                    # cabeza de la Fase 5 del mismo fold
        info = descongelar_pct(model, cfg["pct_capas"])
        bb_params = [p for p in model.backbone.parameters() if p.requires_grad]
        opt = torch.optim.AdamW([dict(params=bb_params, lr=cfg["lr_backbone"]),
                                 dict(params=list(model.head.parameters()), lr=cfg["lr_head"])],
                                weight_decay=cfg["weight_decay"])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
        tr_idx, va_mask = IDX_POOL[FOLD_POOL != k], FOLD_POOL == k
        va_idx, va_ev = IDX_POOL[va_mask], IDX_POOL[va_mask & EVAL_POOL]
        n1 = int(Y[tr_idx].sum())
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(tr_idx) - n1) / max(n1, 1), device=DEVICE))
        tf_ev = build_tf(size, False, model.mean, model.std)
        dl_tr = DataLoader(MapaDS(imgs, tr_idx, build_tf(size, cfg["aug"], model.mean, model.std)),
                           batch_size=cfg["batch"], shuffle=True, num_workers=NUM_WORKERS,
                           generator=torch.Generator().manual_seed(SEED + k))
        ds_va_ev = MapaDS(imgs, va_ev, tf_ev)
        scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        best_auc, best_state, best_ep, wait, hist = -1.0, None, 0, 0, []
        for ep in range(1, cfg["epochs"] + 1):
            model.train()
            ep_loss, n_vistos, t_ep = 0.0, 0, time.time()
            for x, yb in dl_tr:
                x, yb = x.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    loss = crit(model(x).float(), yb)
                if not torch.isfinite(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                ep_loss += loss.item() * len(yb)
                n_vistos += len(yb)
            sched.step()
            p_va = predecir(model, ds_va_ev, size)
            auc = float(roc_auc_score(Y[va_ev], p_va))
            hist.append(dict(epoch=ep, train_loss=round(ep_loss / max(n_vistos, 1), 4),
                             val_loss=round(bce(Y[va_ev], p_va), 4), val_auc=round(auc, 4),
                             s=round(time.time() - t_ep, 1)))
            if best_state is None or auc > best_auc + 1e-4:
                best_auc, best_ep, wait = auc, ep, 0
                best_state = {n: p.detach().cpu().half() for n, p in model.named_parameters() if p.requires_grad}
            else:
                wait += 1
                if wait >= cfg["patience"]:
                    break
        torch.save(best_state, ruta_ckpt(exp_id, k))            # solo lo entrenado, en fp16
        model.load_state_dict({n: v.float() for n, v in best_state.items()}, strict=False)
        oof[va_mask] = predecir(model, MapaDS(imgs, va_idx, tf_ev), size)
        p_test.append(predecir(model, MapaDS(imgs, IDX_TEST, tf_ev), size))
        hist_all[k] = hist
        print(f"  {exp_id} fold {k}: {len(hist)} épocas ({np.mean([h['s'] for h in hist]):.0f} s/época), "
              f"mejor época = {best_ep}, AUC val = {best_auc:.4f}", flush=True)
        liberar(model, opt)
    return armar_res(exp_id, f"fine-tuning {FT_CFG['pct_capas']:.0%} capas", oof, np.mean(p_test, 0),
                     dict(backbone=bb, mapa=mapa, entrada=f"1 imagen por ojo ({mapa})", init=init_id,
                          descongelado=info, historia=hist_all),
                     segundos=time.time() - t0)

In [ ]:
IDS_FT = []
for mapa in MAPAS_FT:
    eid = f"FT_{c}_{mapa}"
    RESULTADOS[eid] = entrenar_ft_cv(eid, BEST_BB, BEST_SIZE, mapa, init_id=f"CAB_{c}_{mapa}")
    IDS_FT.append(eid)
ids_ft_mapas = list(IDS_FT)
if len(MAPAS_FT) > 1:
    eid = f"FT_{c}_fusion"
    RESULTADOS[eid] = fusionar(eid, ids_ft_mapas, f"fine-tuning, fusión de {len(MAPAS_FT)} mapas",
                               dict(backbone=BEST_BB, mapa="fusión",
                                    entrada=f"{len(MAPAS_FT)} imágenes por ojo (promedio de mapas)"))
    IDS_FT.append(eid)

graficar_historia(ids_ft_mapas, f"{c}: avance del fine-tuning (por fold)", f"historia_ft_{c}")
mejor_ft = elegir(tabla(IDS_FT))
fila_cms(IDS_FT, "oof", f"{c} con fine-tuning: matrices de VALIDACIÓN (OOF balanceado)", f"cm_val_ft_{c}", mejor=mejor_ft)
fila_cms(IDS_FT, "test", f"{c} con fine-tuning: matrices de TEST (balanceado)", f"cm_test_ft_{c}", mejor=mejor_ft)

ids_ab = []
for mapa in MAPAS_FT + (["fusion"] if len(MAPAS_FT) > 1 else []):
    ids_ab += [f"CAB_{c}_{mapa}", f"FT_{c}_{mapa}"]
tab_ab = tabla(ids_ab)
tab_ab.insert(1, "etapa", ["antes (TL última capa)" if i.startswith("CAB") else "después (fine-tuning)" for i in tab_ab.ID])
MEJOR_FT_AB = resaltar(tab_ab, ["ID", "etapa", "entrada", "AUC_oof", "Esp_oof", "tau", "AUC_test", "Sens_test",
                                "Esp_test", "BalAcc_test", "Brier_test", "n_test"],
                       titulo="antes / después del fine-tuning")
cerrar_etapa("Fase 6 — Fine-tuning")

## Fase 7 — Embeddings + modelos de ML clásico

Se toma el **vector de características** (*embedding*) que produce cada backbone para cada imagen y
se usa como entrada de clasificadores clásicos, en vez de la última capa de la red.

* **Backbones**: los 3 congelados de la Fase 5 (los mismos vectores cacheados) y el backbone con
  **fine-tuning** de la Fase 6 (sufijo `-FT`). Para respetar la validación cruzada, el embedding
  usado en cada fold sale del modelo FT de **ese** fold, que no vio esos ojos de validación.
* **Entradas**: cada mapa por separado y los 4 mapas **concatenados** (`concat4`).
* **Clasificadores**: regresión logística (L2), SVM RBF (calibrada), Random Forest, XGBoost
  (o HistGradientBoosting si no está instalado) y MLP (una capa oculta de 256).
* **Reducción de dimensión**: SVM, XGBoost y MLP reciben el embedding reducido con **PCA a 128
  componentes**, ajustado solo con el train de cada fold (con 10,240 dimensiones en `concat4`,
  XGBoost y la SVM tardaban 30-100 s por ajuste). Regresión logística y Random Forest usan el
  vector completo.
* Mismos folds, mismo `tau` (sensibilidad OOF ≥ 0.95) y misma evaluación balanceada. El mejor queda
  **en verde** en la tabla y **recuadrado** en el mapa de calor.

In [ ]:
iniciar_etapa("Fase 7 — Embeddings + ML clásico")


class PCAHasta(BaseEstimator, TransformerMixin):
    """PCA a `n` componentes (o menos si hay pocos ojos), ajustado solo con los datos de fit (train del fold)."""
    def __init__(self, n=128):
        self.n = n

    def fit(self, X, y=None):
        self.pca_ = PCA(n_components=min(self.n, X.shape[0] - 1, X.shape[1]), svd_solver="randomized",
                        random_state=SEED).fit(X)
        return self

    def transform(self, X):
        return self.pca_.transform(X)


def make_clasicos():
    pca = lambda: PCAHasta(ML_CFG["pca_comp"])
    d = {"LogReg": lambda: make_pipeline(StandardScaler(),
                                         LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)),
         "SVM": lambda: make_pipeline(StandardScaler(), pca(), CalibratedClassifierCV(
             SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=SEED), cv=3, method="sigmoid")),
         "RandomForest": lambda: RandomForestClassifier(n_estimators=ML_CFG["rf_arboles"], min_samples_leaf=2,
                                                        class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)}
    if XGBClassifier is not None:
        d["XGBoost"] = lambda: make_pipeline(StandardScaler(), pca(), XGBClassifier(
            n_estimators=ML_CFG["xgb_arboles"], max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5,
            eval_metric="logloss", n_jobs=-1, random_state=SEED))
    else:
        d["HistGB"] = lambda: make_pipeline(StandardScaler(), pca(), HistGradientBoostingClassifier(
            max_iter=ML_CFG["xgb_arboles"], learning_rate=0.05, class_weight="balanced", random_state=SEED))
    d["MLP"] = lambda: make_pipeline(StandardScaler(), pca(), MLPClassifier(
        hidden_layer_sizes=(256,), alpha=1e-2, max_iter=500, early_stopping=ML_CFG["mlp_early_stopping"],
        random_state=SEED))
    return d


CLASICOS = make_clasicos()


def emb_ft(mapa):
    """Embeddings del backbone con FT: un juego por fold (del modelo FT de ese fold)."""
    out = {}
    for k in FOLDS:
        f = OUT / "emb" / f"emb_ft_{c}_{mapa}_f{k}{TAG}.npz"
        X = _leer_npz(f)
        if X is None:
            t0 = time.time()
            model = cargar_clf(BEST_BB, f"FT_{c}_{mapa}", k)
            X = emb_modelo(model, imagenes_mapa(mapa, BEST_SIZE), BEST_SIZE)
            _guardar_npz(f, X, t0)
            liberar(model)
        out[k] = X
    return out


IDS_ML = []
variantes = [CORTO[bb] for bb in BACKBONES] + [f"{c}-FT"]
for var in variantes:
    with subetapa(f"Fase 7 · {var}", "Fase 7 — Embeddings + ML clásico"):
        if var.endswith("-FT"):
            E = {mapa: emb_ft(mapa) for mapa in MAPAS_FT}
            entradas = {mapa: (lambda k, m=mapa: E[m][k]) for mapa in MAPAS_FT}
            if len(MAPAS_FT) == len(MAPAS):
                entradas["concat4"] = lambda k: np.concatenate([E[m][k] for m in MAPAS], 1)
        else:
            bb = next(b for b in BACKBONES if CORTO[b] == var)
            entradas = {mapa: EMB[(bb, mapa)] for mapa in MAPAS}
            entradas["concat4"] = np.concatenate([EMB[(bb, m)] for m in MAPAS], 1)
        for ent, X in entradas.items():
            desc = "4 imágenes por ojo (embeddings concatenados)" if ent == "concat4" else f"1 imagen por ojo ({ent})"
            for nom, mk in CLASICOS.items():
                eid = f"ML_{nom}_{var}_{ent}"
                RESULTADOS[eid] = cv_sklearn(eid, f"embeddings + {nom}", X, mk,
                                             dict(backbone=var, mapa=ent, clf=nom, entrada=desc))
                IDS_ML.append(eid)
        print(f"  {var}: {len(entradas) * len(CLASICOS)} combinaciones evaluadas", flush=True)

In [ ]:
tab_ml = tabla(IDS_ML)
tab_ml["clf"] = [RESULTADOS[i]["extra"]["clf"] for i in tab_ml.ID]
tab_ml["col"] = [f"{RESULTADOS[i]['extra']['backbone']}\n{RESULTADOS[i]['extra']['mapa']}" for i in tab_ml.ID]
MEJOR_ML = elegir(tab_ml)
piv = tab_ml.pivot_table(index="clf", columns="col", values="AUC_oof", sort=False)
fig, ax = plt.subplots(figsize=(0.95 * piv.shape[1] + 3, 0.7 * piv.shape[0] + 2.4))
im = ax.imshow(piv.values, cmap="viridis", vmin=max(0.5, np.nanmin(piv.values) - 0.02), vmax=1, aspect="auto")
ax.set_xticks(range(piv.shape[1]), piv.columns, fontsize=7.5)
ax.set_yticks(range(piv.shape[0]), piv.index)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        v = piv.values[i, j]
        ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7.5,
                color="black" if im.norm(v) > 0.6 else "white")
fg = tab_ml.set_index("ID").loc[MEJOR_ML]
ib, jb = list(piv.index).index(fg["clf"]), list(piv.columns).index(fg["col"])
ax.add_patch(Rectangle((jb - 0.5, ib - 0.5), 1, 1, fill=False, ec="#ff2d2d", lw=3))
ax.set_title(f"AUC out-of-fold — embeddings + ML clásico (recuadro rojo = elegido: {MEJOR_ML})", fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout()
plt.savefig(OUT / "fig" / f"auc_ml_clasico{TAG}.png", dpi=110)
plt.show()

resaltar(tab_ml.drop(columns=["clf", "col"]), ["ID", "backbone", "entrada", "AUC_oof", "Esp_oof", "tau", "AUC_test",
                                               "Sens_test", "Esp_test", "BalAcc_test", "n_test"],
         titulo="embeddings + ML clásico", orden=True)
fig, axes = plt.subplots(1, 2, figsize=(7.4, 4.7))
cm_res(axes[0], MEJOR_ML, "oof", titulo=f"{MEJOR_ML}\nVALIDACIÓN", resaltado=True)
cm_res(axes[1], MEJOR_ML, "test", titulo=f"{MEJOR_ML}\nTEST", resaltado=True)
plt.tight_layout()
plt.savefig(OUT / "fig" / f"cm_mejor_ml{TAG}.png", dpi=100)
plt.show()
cerrar_etapa("Fase 7 — Embeddings + ML clásico")

## Fase 8 — Tabla final, elección del modelo y curvas ROC

* Para **cada** experimento, `tau` se calibró con OOF (sensibilidad OOF ≥ 0.95) y se aplicó **sin
  tocar** al test.
* **Regla de elección (la misma de todo el notebook)**: mayor **`AUC_oof`** (redondeado a 3
  decimales); si empatan, mayor **`Esp_oof`** @tau. Solo train/OOF: el test se mira únicamente para
  reportar.
* El **modelo elegido** queda en verde en la tabla, en trazo grueso con ★ en la ROC y con su matriz
  de confusión de validación y de test. El baseline tabular aparece en gris como referencia.

In [ ]:
iniciar_etapa("Fase 8 — Tabla final y ROC")
ORDEN = ["TAB_LR"] + IDS_CAB + IDS_FT + IDS_ML
df_res = tabla(ORDEN)
MEJOR = resaltar(df_res, COLS_FINAL, titulo="MODELO FINAL", orden=True)

orden_sel = (df_res[df_res.ID != "TAB_LR"].assign(_k=df_res[CRITERIO].round(3), _d=df_res[DESEMPATE])
             .sort_values(["_k", "_d"], ascending=False).ID.tolist())
top = list(dict.fromkeys([MEJOR] + orden_sel[:5] + ["TAB_LR"]))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))
ax = axes[0]
for i in top:
    r = RESULTADOS[i]
    y, p = y_p(r, "test")
    fpr, tpr, _ = roc_curve(y, p)
    es = i == MEJOR
    ax.plot(fpr, tpr, lw=3.2 if es else 1.2, color=VERDE_OSC if es else None, ls=":" if i == "TAB_LR" else "-",
            label=f"{'★ ' if es else ''}{i} (AUC test={r['test']['auc']:.3f})", zorder=4 if es else 2)
    ax.scatter([1 - r["test"]["esp"]], [r["test"]["sens"]], s=160 if es else 35, marker="*" if es else "o",
               color=VERDE_OSC if es else None, zorder=6 if es else 5, edgecolor="k" if es else None)
ax.axhline(OBJ_SENS, ls="--", color="gray")
ax.plot([0, 1], [0, 1], ":", color="lightgray")
ax.set_xlabel("1 - especificidad")
ax.set_ylabel("sensibilidad (recall QC)")
ax.set_title("ROC en test (balanceado) — puntos = operación con el tau de OOF")
ax.legend(fontsize=7, loc="lower right")

ax = axes[1]
x = np.arange(len(top))
sens = [RESULTADOS[i]["test"]["sens"] for i in top]
sens_err = np.clip(np.array([[s - RESULTADOS[i]["test"]["sens_ic"][0] for s, i in zip(sens, top)],
                             [RESULTADOS[i]["test"]["sens_ic"][1] - s for s, i in zip(sens, top)]]), 0, None)
b1 = ax.bar(x - 0.2, sens, 0.4, yerr=sens_err, capsize=3, color="tab:red", label="sensibilidad @tau (IC Wilson)")
b2 = ax.bar(x + 0.2, [RESULTADOS[i]["test"]["esp"] for i in top], 0.4, color="tab:blue", label="especificidad @tau")
ax.legend(fontsize=8, loc="lower right")          # antes de marcar el elegido, así la leyenda no hereda el borde
for bars in (b1, b2):
    bars[0].set_edgecolor(VERDE_OSC)
    bars[0].set_linewidth(3)
ax.axhline(OBJ_SENS, ls="--", color="gray")
ax.set_xticks(x, [("★ " if i == MEJOR else "") + i for i in top], rotation=40, ha="right", fontsize=8)
ax.get_xticklabels()[0].set_color(VERDE_OSC)
ax.get_xticklabels()[0].set_fontweight("bold")
ax.set_ylim(0, 1.05)
ax.set_title("Punto de operación en test (★ = elegido)")
plt.tight_layout()
plt.savefig(OUT / "fig" / f"roc_operacion{TAG}.png", dpi=110)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11, 4.7))
cm_res(axes[0], MEJOR, "oof", titulo=f"{MEJOR}\nVALIDACIÓN", resaltado=True)
cm_res(axes[1], MEJOR, "test", titulo=f"{MEJOR}\nTEST", resaltado=True)
cm_res(axes[2], "TAB_LR", "test", titulo="TAB_LR (referencia)\nTEST")
plt.tight_layout()
plt.savefig(OUT / "fig" / f"cm_modelo_final{TAG}.png", dpi=100)
plt.show()
cerrar_etapa("Fase 8 — Tabla final y ROC")

## Fase 9 — Explicabilidad: Grad-CAM

### Cómo leer los colores
Grad-CAM produce un **mapa de calor** del tamaño de la imagen que indica **cuánto contribuyó cada
zona a la decisión del modelo** (se calcula con los gradientes sobre la última capa convolucional
del backbone):

* 🔴 **rojo / amarillo** → zonas que **más empujaron** la predicción: ahí "se fijó" el modelo;
* 🟢 verde / celeste → contribución intermedia;
* 🔵 **azul oscuro** → zonas que **casi no influyeron**.

Es decir: rojo = donde mira, azul = donde no mira. Hay que tener en cuenta dos cosas:

1. **Se explica la clase que el modelo predijo** (con su `tau`): si predijo QC, el rojo es la
   evidencia de QC; si predijo sano, el rojo es la evidencia de "sano". Grad-CAM muestra solo lo que
   suma a favor de esa clase; lo que resta queda en azul.
2. **Cada mapa se normaliza a su propio máximo** (0 = nada, 1 = la zona más importante *de esa
   imagen*). Siempre aparece algo en rojo, aunque la evidencia total sea débil: no compares la
   intensidad entre imágenes, mirá **la ubicación**.

Como los mapas Orbscan **ya usan colores parecidos** (rojo = más curvo, más delgado o más elevado),
el mapa de calor se muestra **solo** (con su escala) y además **superpuesto**. Si el rojo cae sobre
los **números impresos** del Axial o de la Paquimetría, y no sobre la zona de encurvamiento o
adelgazamiento, el modelo está usando un atajo.

In [ ]:
iniciar_etapa("Fase 9 — Grad-CAM")
# ejemplo visual de la escala de colores (imagen sintética)
yy, xx = np.mgrid[0:200, 0:200]
img_sint = np.full((200, 200, 3), 0.92)
disco = (xx - 100) ** 2 + (yy - 100) ** 2 < 85 ** 2
img_sint[disco] = [0.55, 0.75, 0.55]
cam_sint = np.exp(-(((xx - 125) ** 2 + (yy - 135) ** 2) / (2 * 22 ** 2)))
cam_sint += 0.35 * np.exp(-(((xx - 70) ** 2 + (yy - 70) ** 2) / (2 * 30 ** 2)))
cam_sint /= cam_sint.max()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4), constrained_layout=True)
axes[0].imshow(img_sint)
axes[0].set_title("1) imagen de entrada (ejemplo)")
h = axes[1].imshow(cam_sint, cmap="jet", vmin=0, vmax=1)
axes[1].annotate("máxima contribución\n(el modelo mira acá)", (125, 135), (150, 20), color="white", fontsize=9,
                 ha="center", arrowprops=dict(color="white", arrowstyle="->"))
axes[1].annotate("contribución media", (70, 70), (55, 185), color="white", fontsize=9, ha="center",
                 arrowprops=dict(color="white", arrowstyle="->"))
axes[1].set_title("2) mapa de calor de Grad-CAM, solo")
axes[2].imshow(img_sint)
axes[2].imshow(cam_sint, cmap="jet", alpha=0.45, vmin=0, vmax=1)
axes[2].set_title("3) superpuesto sobre la imagen")
for ax in axes:
    ax.axis("off")
cb = fig.colorbar(h, ax=axes, orientation="horizontal", fraction=0.07, pad=0.04, shrink=0.8, aspect=45)
cb.set_ticks([0, 0.5, 1], labels=["0\nazul: casi no influyó", "0.5\nintermedio",
                                  "1\nrojo: lo que más empujó la predicción"])
fig.suptitle("Cómo leer Grad-CAM (normalizado por imagen: 1 = la zona más importante de esa imagen)", fontsize=12)
plt.show()

In [ ]:
def gradcam(model, x, clase):
    """Grad-CAM de la clase `clase` (1 = QC: se deriva el logit; 0 = sano: se deriva -logit)."""
    model.eval()
    x = x.clone().requires_grad_(True)
    feats = model.backbone.forward_features(x)
    feats.retain_grad()
    logit = model.head(model.backbone.forward_head(feats, pre_logits=True)).squeeze()
    objetivo = logit if clase == 1 else -logit
    model.zero_grad(set_to_none=True)
    objetivo.backward()
    w = feats.grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * feats).sum(1))[0].detach().float().cpu().numpy()
    vacio = cam.max() < 1e-8                 # ninguna zona suma a favor de la clase -> mapa todo azul
    return cam / (cam.max() + 1e-8), float(torch.sigmoid(logit).item()), vacio


idx_te_ev = IDX_TEST[TEST_BAL]
rng = np.random.default_rng(SEED)
sel_cam = np.concatenate([rng.choice(idx_te_ev[Y[idx_te_ev] == lab], GRADCAM_POR_CLASE, replace=False)
                          for lab in (1, 0)])
k0 = FOLDS[0]
for mapa in MAPAS_FT:
    ids_m = [("TL última capa", f"CAB_{c}_{mapa}"), ("fine-tuning", f"FT_{c}_{mapa}")]
    modelos = [(nom, eid, cargar_clf(BEST_BB, eid, k0)) for nom, eid in ids_m]
    imgs = imagenes_mapa(mapa, BEST_SIZE)
    tf_ev = build_tf(BEST_SIZE, False, modelos[0][2].mean, modelos[0][2].std)
    fig, axes = plt.subplots(len(sel_cam), 5, figsize=(17, 3.6 * len(sel_cam)), squeeze=False,
                             constrained_layout=True)
    for fi, j in enumerate(sel_cam):
        vis = np.asarray(T.CenterCrop(BEST_SIZE)(imgs[j]))
        axes[fi, 0].imshow(vis)
        axes[fi, 0].set_title(f"{man.eye_key[j]} | real = {'QC' if Y[j] else 'sano'}", fontsize=9)
        for ci, (nom, eid, mdl) in enumerate(modelos):
            tau_m = RESULTADOS[eid]["oof"]["tau"]
            with torch.no_grad():
                p0 = float(torch.sigmoid(mdl(tf_ev(imgs[j]).unsqueeze(0).to(DEVICE))).item())
            pred = int(p0 >= tau_m)
            cam, prob, vacio = gradcam(mdl, tf_ev(imgs[j]).unsqueeze(0).to(DEVICE), pred)
            cam_img = np.asarray(Image.fromarray(np.uint8(255 * cam)).resize((BEST_SIZE, BEST_SIZE), Image.BILINEAR)) / 255
            ok = "✓" if pred == Y[j] else "✗"
            hm = axes[fi, 1 + 2 * ci].imshow(cam_img, cmap="jet", vmin=0, vmax=1)
            axes[fi, 1 + 2 * ci].set_title(f"{nom}: calor (explica '{'QC' if pred else 'sano'}')"
                                           + ("\nninguna zona suma a favor" if vacio else ""), fontsize=9)
            axes[fi, 2 + 2 * ci].imshow(vis)
            axes[fi, 2 + 2 * ci].imshow(cam_img, cmap="jet", alpha=0.45, vmin=0, vmax=1)
            axes[fi, 2 + 2 * ci].set_title(f"{nom}: pred = {'QC' if pred else 'sano'} {ok} · P(QC) = {prob:.2f}",
                                           fontsize=9)
        for ax in axes[fi]:
            ax.axis("off")
    cb = fig.colorbar(hm, ax=axes, orientation="vertical", fraction=0.015, pad=0.01)
    cb.set_ticks([0, 0.5, 1], labels=["0 azul: no influyó", "0.5", "1 rojo: más influyó"])
    fig.suptitle(f"Grad-CAM — {mapa} ({c}, fold {k0}): original | TL última capa | fine-tuning", fontsize=12)
    plt.savefig(OUT / "fig" / f"gradcam_{mapa}{TAG}.png", dpi=80)
    plt.show()
    liberar(*[m for _, _, m in modelos])
    del modelos
cerrar_etapa("Fase 9 — Grad-CAM")

## Fase 10 — Resumen, exportación y tiempos de ejecución

In [ ]:
b, t, o = RESULTADOS[MEJOR]["test_boot"], RESULTADOS[MEJOR]["test"], RESULTADOS[MEJOR]["oof"]
tb = RESULTADOS["TAB_LR"]["test"]
frac_txt = "SMOKE" if SMOKE else f"{min(FRACCION_DATOS, 1):.0%} de los pacientes"
lineas = [
    f"### Resultados — análisis de backbones (queratocono){' — SMOKE' if SMOKE else ''}",
    "",
    f"* Datos: **{frac_txt}** → {len(man)} ojos; train {len(IDX_POOL)} (CV {len(FOLDS)} folds oficiales) / "
    f"test {len(IDX_TEST)}. Evaluación balanceada: OOF {int(EVAL_POOL.sum())} ojos, test {int(TEST_BAL.sum())} "
    f"({t['n0']} sanos / {t['n1']} QC). Imágenes {'sin' if LIMPIAR_MARCAS else 'con'} letras T/N.",
    f"* Criterio de elección: `{CRITERIO}` (3 decimales), desempate `{DESEMPATE}`. Objetivo: sensibilidad OOF ≥ {OBJ_SENS}.",
    f"* **Mejor transfer learning (Fase 5)**: `{MEJOR_CAB}` → backbone que pasó al fine-tuning: `{BEST_BB}`.",
    f"* **Mejor ML clásico (Fase 7)**: `{MEJOR_ML}`.",
    f"* **Modelo final elegido por OOF**: `{MEJOR}` → tau = {o['tau']:.3f}; en test: "
    f"sensibilidad = {t['sens']:.3f} [Wilson {t['sens_ic'][0]:.3f}-{t['sens_ic'][1]:.3f}], "
    f"especificidad = {t['esp']:.3f} [boot {b['esp'][0]:.3f}-{b['esp'][1]:.3f}], "
    f"AUC = {t['auc']:.3f} [boot {b['auc'][0]:.3f}-{b['auc'][1]:.3f}], FN = {t['fn']}, FP = {t['fp']}.",
    f"* **Baseline tabular** (referencia, sin imágenes): AUC test = {tb['auc']:.3f}, sensibilidad = {tb['sens']:.3f}, "
    f"especificidad = {tb['esp']:.3f}.",
    "* Grad-CAM: figuras en `fig/gradcam_*.png`. Su interpretación (mapa frente a números impresos) se redacta a mano.",
    f"* Limitaciones: el test balanceado tiene {t['n1']} QC, así que el IC de la sensibilidad es amplio y las diferencias chicas no son concluyentes.",
]
txt = "\n".join(lineas)
display(Markdown(txt))

csv_out = OUT / f"02_resultados_backbone{TAG}.csv"
df_res.to_csv(csv_out, index=False)
(OUT / f"02_resumen{TAG}.md").write_text(txt + "\n", encoding="utf-8")
print("CSV:", csv_out.resolve())
print("resumen:", (OUT / f"02_resumen{TAG}.md").resolve())

### Tiempos de ejecución por etapa

Tiempo de cada fase (y, dentro de las fases 5 y 7, de cada backbone). Si algo se recuperó de la
caché (porque ya se había calculado en una corrida anterior), se suma el tiempo que había costado
calcularlo, para que la tabla refleje el costo real de la etapa. El **total** suma solo las fases.

In [ ]:
padres = [n for n, d in TIEMPOS.items() if not d.get("padre")]
hijos = {}
for n, d in TIEMPOS.items():
    if d.get("padre"):
        hijos.setdefault(d["padre"], []).append(n)
filas_t = []
for n in padres:
    d = TIEMPOS[n]
    filas_t.append({"Etapa": n, "Tiempo": fmt_t(d["total_s"]), "segundos": d["total_s"],
                    "de caché (s)": d["cache_s"], "% del total": None})
    for h in hijos.get(n, []):
        dh = TIEMPOS[h]
        filas_t.append({"Etapa": f"      ↳ {h}", "Tiempo": fmt_t(dh["total_s"]), "segundos": dh["total_s"],
                        "de caché (s)": dh["cache_s"], "% del total": None})
total = sum(TIEMPOS[n]["total_s"] for n in padres)
for f_ in filas_t:
    if not f_["Etapa"].startswith(" "):
        f_["% del total"] = round(100 * f_["segundos"] / total, 1) if total else 0.0
filas_t.append({"Etapa": "TOTAL", "Tiempo": fmt_t(total), "segundos": round(total, 1),
                "de caché (s)": round(sum(TIEMPOS[n]["cache_s"] for n in padres), 1), "% del total": 100.0})
df_t = pd.DataFrame(filas_t)
display(df_t.style.apply(lambda r: ["font-weight:bold;background-color:#eeeeee" if r["Etapa"] == "TOTAL" else
                                    ("color:#666666" if r["Etapa"].startswith(" ") else "") for _ in r], axis=1)
        .format({"segundos": "{:.1f}", "de caché (s)": "{:.1f}", "% del total": lambda v: "" if pd.isna(v) else f"{v:.1f}"})
        .hide(axis="index"))
df_t.to_csv(OUT / f"tiempos{TAG}.csv", index=False)

d_pad = df_t[~df_t.Etapa.str.startswith(" ") & (df_t.Etapa != "TOTAL")]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(d_pad) + 1.2))
ax.barh(d_pad.Etapa[::-1], d_pad.segundos[::-1] / 60, color="tab:gray")
for i, (s, tx) in enumerate(zip(d_pad.segundos[::-1], d_pad.Tiempo[::-1])):
    ax.text(s / 60, i, f" {tx}", va="center", fontsize=8)
ax.set_xlabel("minutos")
ax.set_title(f"Tiempo por etapa — total {fmt_t(total)}")
plt.tight_layout()
plt.savefig(OUT / "fig" / f"tiempos{TAG}.png", dpi=100)
plt.show()
print("tiempos:", (OUT / f"tiempos{TAG}.csv").resolve())

## Reproducibilidad

* **Semilla 42** en el subset, la evaluación balanceada, los DataLoaders (`generator`), cada fold (`SEED + k`), los clasificadores clásicos y Grad-CAM.
* **Splits oficiales** del dataset (`split_80_20`, `fold`). Se verifica por código que no hay paciente en dos folds ni en train y test a la vez, también dentro del subset.
* **Qué se calibra con qué**: la selección de época, backbone y modelo, y el umbral `tau`, usan **solo OOF/validación** (balanceada). El test (promedio de los modelos de cada fold, balanceado) solo se usa para reportar.
* **Imágenes**: `ORBSCAN_Dataset_modificado` (letras T/N tapadas con un rectángulo blanco fijo); el original no se modifica. `LIMPIAR_MARCAS = False` vuelve a las originales (caché aparte, sufijo `_conTN`).
* **Caché**: `res/*.json` (probabilidades, claves de ojo y tiempo de cómputo), `ckpt/*.pth` (cabeza y capas liberadas por el FT, en fp16) y `emb/*.npz`. Si existen, no se recalculan. En Colab viven en Drive y sobreviven al corte de sesión.
* **Smoke test**: `SMOKE=1 jupyter nbconvert --to notebook --execute 02_Analisis_BackBone.ipynb --output 02_smoke_out.ipynb`.